In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 72.3 MB/s eta 0:00:00


In [ ]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.1 MB/s eta 0:00:00


In [ ]:
import os

# 2. Tạo thư mục lưu trữ (bạn có thể đổi tên thư mục tùy ý)
save_path = "/content/drive/MyDrive/Qwen_Model_3B"
os.makedirs(save_path, exist_ok=True)

# 3. Cài đặt thư viện tải file
!pip install -q huggingface_hub

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

class MeetingSummarizer:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # ID của mô hình Qwen2.5-3B-Instruct chính thức
        model_id = "Qwen/Qwen2.5-3B-Instruct"

        print(f"Đang tải model {model_id} với cấu hình 4-bit...")

        # Cấu hình lượng tử hóa 4-bit để tiết kiệm VRAM
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config, # Nạp với cấu hình 4-bit
            device_map="auto"
        )

        self.model.eval()
        print("Model 3B đã sẵn sàng hoạt động!")

In [ ]:
from huggingface_hub import snapshot_download

print("Bắt đầu tải mô hình về Drive (quá trình này mất vài phút)...")

snapshot_download(
    repo_id="Qwen/Qwen2.5-3B-Instruct",
    local_dir=save_path,
    local_dir_use_symlinks=False # Quan trọng để file thực sự nằm trong Drive
)

print(f"Hoàn tất! Mô hình đã nằm tại: {save_path}")

Bắt đầu tải mô hình về Drive (quá trình này mất vài phút)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Hoàn tất! Mô hình đã nằm tại: /content/drive/MyDrive/Qwen_Model_3B


hoanchinh

In [ ]:
import torch
import re
import logging
import networkx as nx
from transformers import AutoTokenizer, AutoModelForCausalLM
from underthesea import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BitsAndBytesConfig
logging.basicConfig(level=logging.ERROR)

class MeetingSummarizer:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Device đang sử dụng:", self.device)

        model_path = "/content/drive/MyDrive/Qwen_Model_3B"

        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()
        print("Model sẵn sàng hoạt động!\n")

    def clean_text(self, text):
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"\n+", " ", text)
        return text.strip()

    def generate_summary_batch(self, texts, max_new_tokens=200, is_final=False):
        if is_final:
            system_prompt = """Bạn là Thư ký cuộc họp. Hãy tóm tắt văn bản thành danh sách gạch đầu dòng phẳng.
<rules>
1. Bắt đầu mỗi ý bằng ký tự "-". KHÔNG dán nhãn chủ đề.
2. KHÔNG ghi các từ mào đầu như: "Vấn đề:", "Ý kiến:", "Kết luận:".
3. TRÌNH TỰ THỜI GIAN: Tóm tắt bao quát toàn bộ sự kiện theo đúng thứ tự.
4. TRÍCH XUẤT TRỰC TIẾP.
</rules>"""
            rep_penalty = 1.05
        else:
            system_prompt = """Trích xuất ý chính từ đoạn hội thoại.
<rules>
- Viết dưới dạng gạch đầu dòng (-).
- LỌC RÁC CỰC MẠNH.
- Nếu toàn rác -> PASS
</rules>"""
            rep_penalty = 1.05
        prompts = []

        for text in texts:
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"<van_ban>\n{text}\n</van_ban>\n\nHãy tóm tắt:"}
            ]
            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            prompts.append(prompt)
        inputs = self.tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2500
        ).to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens if is_final else 200,
                repetition_penalty=rep_penalty,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )
        results = []
        for i in range(len(prompts)):
            generated_ids = outputs[i][len(inputs.input_ids[i]):]
            summary = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
            summary = summary.replace("<|im_end|>", "").strip()
            results.append(summary)
        return results

    def generate_summary(self, text, max_new_tokens=1000, is_final=True):
        if is_final:
            system_prompt = """Bạn là Thư ký cuộc họp. Hãy tóm tắt văn bản thành danh sách gạch đầu dòng phẳng.
<rules>
1. Bắt đầu mỗi ý bằng ký tự "-". KHÔNG dán nhãn chủ đề.
2. KHÔNG ghi các từ mào đầu như: "Vấn đề:", "Ý kiến:", "Kết luận:".
3. TRÌNH TỰ THỜI GIAN: Tóm tắt bao quát toàn bộ sự kiện theo đúng thứ tự.
4. TRÍCH XUẤT TRỰC TIẾP.
</rules>"""
            rep_penalty = 1.05
        else:
            system_prompt = """Trích xuất ý chính từ đoạn hội thoại.
<rules>
- Viết dưới dạng gạch đầu dòng (-).
- LỌC RÁC CỰC MẠNH.
- Nếu toàn rác -> PASS
</rules>"""
            rep_penalty = 1.05

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"<van_ban>\n{text}\n</van_ban>\n\nHãy tóm tắt:"}
        ]

        text_input = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text_input, return_tensors="pt", truncation=True, max_length=2500).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens if is_final else 200,
                repetition_penalty=rep_penalty,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )

        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, outputs)]
        summary = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        summary = summary.replace("<|im_end|>", "").strip()
        if not is_final:
            return summary

        cleaned_lines = []
        for line in summary.split('\n'):
            line = line.strip()
            if not line or re.search(r'(chào các bạn|chào mọi người|cảm ơn|hẹn gặp lại)', line, re.IGNORECASE):
                continue
            line = re.sub(r"^(- )?(### Biên bản|Biên bản họp|Tóm tắt|Nội dung chính|Vấn đề).*?:?", "", line, flags=re.IGNORECASE).strip()
            if line == "-" or line == "":
                continue
            if line:
                if not line.startswith('-'):
                    line = '- ' + line
                cleaned_lines.append(line)
        final_summary = '\n'.join(cleaned_lines).strip()
        if not final_summary:
            print("[Warning] Model bị ngắt kết quả, kích hoạt Fallback chia câu.")
            sentences = sent_tokenize(text)
            return "\n".join(["- " + s for s in sentences])
        return final_summary


    def chunking_method(self, text):
        sentences = sent_tokenize(text)
        if len(sentences) < 6:
            return self.generate_summary(text, is_final=True)
        chunk_size = 10
        stride = 7
        chunks = []
        for i in range(0, len(sentences), stride):
            chunk = " ".join(sentences[i : i + chunk_size])
            chunks.append(chunk)
            if i + chunk_size >= len(sentences):
                break
        batch_outputs = self.generate_summary_batch(
            chunks,
            max_new_tokens=200,
            is_final=False
        )
        partials = []
        for part_sum in batch_outputs:
            if "PASS" not in part_sum and len(part_sum) > 10:
                partials.append(part_sum)

        merged = "\n".join(partials)

        flat_summary = self.generate_summary(merged, max_new_tokens=1000, is_final=True)
        # final_grouped_paragraphs = self.dynamic_clustering_to_paragraphs(flat_summary)
        # return final_grouped_paragraphs
        return flat_summary

    def enhanced_textrank(self, text):
        sentences = sent_tokenize(text)
        if len(sentences) < 4:
            return self.generate_summary(text, is_final=True)

        sentences_seg = [word_tokenize(s, format="text") for s in sentences]
        try:
            tfidf = TfidfVectorizer().fit_transform(sentences_seg)
            sim_matrix = cosine_similarity(tfidf)
            graph = nx.from_numpy_array(sim_matrix)
            scores = nx.pagerank(graph)

            ranked = sorted(((scores[i], s) for i, s in enumerate(sentences)), reverse=True)
            top_n = max(4, int(len(sentences) * 0.45))

            selected_sentences_set = set([s for _, s in ranked[:top_n]])

            for initial_sentence in sentences[:2]:
                selected_sentences_set.add(initial_sentence)

            selected_sentences = sorted(list(selected_sentences_set), key=lambda x: sentences.index(x))
            filtered_text = " ".join(selected_sentences)

            flat_summary = self.generate_summary(filtered_text, is_final=True)
            # final_grouped_paragraphs = self.dynamic_clustering_to_paragraphs(flat_summary)
            # return final_grouped_paragraphs
            return flat_summary

        except Exception as e:
            print(f"Lỗi TextRank: {e}")
            flat_summary = self.generate_summary(text, is_final=True)
            # return self.dynamic_clustering_to_paragraphs(flat_summary)
            return flat_summary

    def dynamic_clustering_to_paragraphs(self, flat_bullets):
        system_prompt = """Bạn là Chuyên gia soạn thảo văn bản hành chính.

Nhiệm vụ KHÔNG phải tóm tắt.

Hãy đọc danh sách các sự kiện và:
1. Tự nhận diện số lượng chủ đề phù hợp với nội dung (có thể là 1 hoặc nhiều chủ đề, không giới hạn cứng)
2. Gom các ý liên quan vào cùng chủ đề
3. Viết lại thành đoạn văn mạch lạc, có liên kết logic

QUAN TRỌNG:
- KHÔNG được làm mất bất kỳ ý nào
- KHÔNG được tóm tắt ngắn lại
- KHÔNG được bỏ sót thông tin
- Nếu có ý trùng → gộp nhưng phải giữ đủ nội dung
- KHÔNG thêm thông tin ngoài danh sách

GỢI Ý:
- Nếu tất cả nội dung cùng một vấn đề → chỉ tạo 1 chủ đề
- Nếu nội dung tách biệt rõ → chia thành nhiều chủ đề tương ứng

Định dạng:
[TÊN CHỦ ĐỀ]:
(đoạn văn đầy đủ, giữ toàn bộ thông tin)
"""
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{flat_bullets}"}
        ]

        text_input = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        inputs = self.tokenizer(text_input, return_tensors="pt", truncation=True, max_length=2500).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=1000,
                repetition_penalty=1.1,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, outputs)]
        final_text = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return final_text.replace("<|im_end|>", "").strip()

In [ ]:
# Đoạn code chạy Test
if __name__ == "__main__":
    summarizer = MeetingSummarizer()



Device đang sử dụng: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model sẵn sàng hoạt động!



In [ ]:
!pip install rouge-score numpy

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=468733279ec5ec59a861433a3e880addaca89fd32feba8069099777a2b440619
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
# 1. DỮ LIỆU KIỂM THỬ (Đã được chuẩn hóa định dạng chuỗi của Python)
test_data = [
    # --- 10 MẪU NGẮN ---
    {
        "transcript": "Chào sếp và anh chị chúng ta bắt đầu hội ý nhanh để cập nhật tình hình sáng nay. Tiến độ dự án hiện tại team đã hoàn thành 78 phần trăm theo kế hoạch đề ra từ đầu tuần. Cụ thể module đăng ký người dùng đã xong phần backend kết nối database mượt mà và frontend giao diện người dùng chạy ổn định trên các thiết bị test. Hôm nay chúng ta sẽ tập trung vào việc test full flow người dùng để phát hiện và fix hai bug nhỏ về validation input. Phần thiết kế giao diện wireframe đã được áp dụng toàn bộ feedback từ phía khách hàng nên tiến độ đạt 72 phần trăm mà không gặp bất kỳ trở ngại kỹ thuật nào. Dự án vẫn đang đi đúng timeline và không có rủi ro delay. Anh chuẩn bị bản demo chi tiết lúc 15 giờ chiều nay để cả team cùng review và góp ý. Chị gửi file thiết kế mới nhất trước 14 giờ để anh integrate vào code chính. Ok tôi sẽ chuẩn bị demo đầy đủ với các case test. Tôi gửi file ngay sau khi họp xong luôn cho mọi người tiện kiểm tra. Tốt lắm dự án vẫn giữ được nhịp độ tốt như vậy. Cảm ơn anh chị đã phối hợp chặt chẽ trong tuần qua. Kết thúc hội ý nhanh ở đây và chúc cả team một ngày làm việc hiệu quả.",
        "reference": "[TIẾN ĐỘ DỰ ÁN]:\nĐội ngũ đã cập nhật tiến độ sáng nay đạt 78 phần trăm tổng thể theo kế hoạch. Toàn team tập trung hoàn thiện testing full flow và fix hai bug nhỏ còn lại trong ngày hôm nay trong khi anh chuẩn bị demo lúc 15 giờ và chị gửi file thiết kế cập nhật trước 14 giờ để integrate kịp deadline cuối ngày"
    },
    {
        "transcript": "Chào sếp chúng ta cần xin duyệt gấp chi phí mua license phần mềm quản lý dự án mới vì bản hiện tại hết hạn tuần sau. Chi phí là 5 triệu 2 trăm nghìn đồng cho gói 6 tháng giúp team theo dõi task realtime và báo cáo tự động. Phần mềm này sẽ cải thiện hiệu suất phối hợp giữa các bộ phận bằng cách sync dữ liệu ngay lập tức và giảm thời gian họp hành. Ok chi phí hợp lý và cần thiết cho dự án hiện tại đang có nhiều task song song. Anh tiến hành mua ngay trong ngày hôm nay sau khi được duyệt để tránh gián đoạn công việc. Chị sẽ hỗ trợ kiểm tra hóa đơn và gửi cho kế toán thanh toán trước cuối tuần. Đồng ý tôi mua xong gửi link license cho toàn team lúc 11 giờ sáng. Tôi test thử ngay khi có để đảm bảo mọi người sử dụng mượt mà từ chiều nay. Tốt chúng ta không cần họp dài làm luôn đi để kịp tiến độ. Cảm ơn sếp đã duyệt nhanh. Kết thúc hội ý nhanh ở đây.",
        "reference": "[DUYỆT CHI PHÍ KHẨN]:\nSếp đã duyệt chi phí mua license phần mềm quản lý dự án với số tiền 5 triệu 2 trăm nghìn đồng cho gói 6 tháng. Anh tiến hành mua ngay trong ngày hôm nay và gửi hóa đơn cho kế toán để thanh toán trước cuối tuần trong khi chị hỗ trợ kiểm tra và cập nhật link license cho toàn team."
    },
    {
        "transcript": "Chào sếp và anh chị chúng ta chốt màu cho logo và giao diện hôm nay. Tôi đề xuất xanh lá đậm HEX #00A651 kết hợp trắng để trông tươi mới năng động và dễ nhìn trên cả web lẫn mobile. Màu này khớp hoàn toàn với brief khách hàng và nổi bật hơn so với xanh dương trước đó. Ok anh thấy màu xanh lá đậm phù hợp vì nó truyền tải sự phát triển và thân thiện. Sếp cũng đồng ý màu này sẽ giúp thương hiệu nổi bật hơn trong thị trường. Chúng ta chốt xanh lá đậm chính thức làm màu chủ đạo. Chị áp dụng ngay vào toàn bộ file thiết kế bao gồm logo icon và background. Anh review code liên quan sau khi nhận file. Đồng ý tôi sẽ chỉnh toàn bộ và gửi bản update cho mọi người. Tôi gửi file cuối cùng trước 16 giờ để kịp deadline. Tốt không thay đổi gì nữa. Cảm ơn anh chị đã góp ý nhanh. Kết thúc hội ý ngắn gọn ở đây.",
        "reference": "[CHỐT MÀU THIẾT KẾ]:\nĐội ngũ thống nhất chọn màu xanh lá đậm mã HEX #00A651 kết hợp trắng cho bộ nhận diện thương hiệu. Chị áp dụng ngay vào file thiết kế và gửi bản cuối cùng trước 16 giờ hôm nay để anh review code liên quan."
    },
    {
        "transcript": "Chào anh chị chúng ta thông báo nhanh lịch nghỉ lễ sắp tới. Công ty nghỉ lễ Giỗ Tổ Hùng Vương hai ngày 29 và 30 tháng 4. Toàn đội làm việc bình thường đến hết ngày 28 tháng 4 để hoàn tất hết task quan trọng. Sau đó nghỉ hai ngày và quay lại làm việc vào ngày 2 tháng 5 như bình thường. Anh gửi email nhắc nhở chi tiết cho khách hàng hôm nay để họ biết lịch và không đặt meeting trong thời gian nghỉ. Chị cũng bàn giao hết file đang dang dở cho khách trước khi nghỉ để tránh delay. Ok tôi soạn email nhắc nhở ngay và gửi trước 12 giờ. Tôi hỗ trợ anh nếu cần thêm nội dung chi tiết về lịch. Đồng ý chúng ta giữ nhịp làm việc đến ngày 28 tháng 4. Tốt không có thay đổi gì khác. Cảm ơn anh chị đã lưu ý lịch nghỉ. Kết thúc hội ý nhanh ở đây.",
        "reference": "[THÔNG BÁO LỊCH NGHỈ LỄ]:\nCông ty nghỉ lễ Giỗ Tổ Hùng Vương ngày 29 tháng 4 và 30 tháng 4. Toàn đội làm việc bình thường đến hết ngày 28 tháng 4 sau đó nghỉ hai ngày và trở lại vào ngày 2 tháng 5. Anh gửi lịch nhắc nhở chi tiết cho khách hàng trước ngày mai để đảm bảo không ảnh hưởng tiến độ dự án."
    },
    {
        "transcript": "Chào sếp và anh chị khách hàng đề nghị họp online thứ Sáu tuần này. Tôi gợi ý 10 giờ sáng để phù hợp múi giờ cả hai bên và không ảnh hưởng lịch nội bộ. Phần slide tiến độ và agenda tôi sẽ chuẩn bị đầy đủ gửi cho mọi người xem trước ngày mai. Ok chị có thể demo phần giao diện đã chốt trong cuộc họp để khách thấy rõ. Sếp tham gia 15 phút đầu để chào và giới thiệu tổng quan dự án. Đồng ý 10 giờ sáng thứ Sáu là ổn nhất. Anh confirm lịch với khách ngay hôm nay và cập nhật vào lịch chung cho team. Chị hỗ trợ demo và trả lời phần thiết kế nếu khách hỏi. Tôi chuẩn bị slide chi tiết với dữ liệu mới nhất. Tốt chúng ta sẽ có cuộc họp hiệu quả. Cảm ơn anh chị đã phối hợp nhanh. Kết thúc hội ý ngắn ở đây.",
        "reference": "[PHỐI HỢP LỊCH HỌP KHÁCH HÀNG]:\nĐội ngũ chốt lịch họp online với khách hàng vào 10 giờ sáng thứ Sáu tuần này. Anh chuẩn bị slide tiến độ và agenda chị hỗ trợ demo phần giao diện và sếp tham gia 15 phút đầu để giới thiệu."
    },
    {
        "transcript": "Chào sếp và anh chị chúng ta bắt đầu hội ý nhanh để cập nhật tình hình sáng nay. Tiến độ test QA tuần này team đã hoàn thành 85 phần trăm theo kế hoạch. Cụ thể module thanh toán đã test xong 120 case bao gồm cả edge case và tích hợp với cổng VNPay chạy ổn định. Hôm nay chúng ta sẽ tập trung chạy regression test cho phần giỏ hàng để phát hiện và fix hai bug về hiển thị giá khuyến mãi. Phần báo cáo tự động đã áp dụng toàn bộ feedback từ khách hàng nên tiến độ đạt 81 phần trăm mà không gặp trở ngại nào. Dự án vẫn đúng timeline và không có rủi ro delay. Anh chuẩn bị bản test report chi tiết lúc 14 giờ chiều nay để cả team review. Chị gửi danh sách case test bổ sung trước 13 giờ để anh cập nhật vào tool. Ok tôi sẽ chạy full regression và gửi kết quả ngay sau khi họp xong. Tôi test thử thêm trên môi trường staging luôn cho chắc. Tốt lắm team giữ được nhịp độ cao như vậy. Cảm ơn anh chị đã phối hợp chặt chẽ. Kết thúc hội ý nhanh ở đây và chúc cả team một ngày làm việc hiệu quả.",
        "reference": "[CẬP NHẬT TIẾN ĐỘ TEST QA]:\nĐội ngũ đã cập nhật tiến độ test QA sáng nay đạt 85 phần trăm tổng thể theo kế hoạch. Toàn team tập trung chạy regression test cho module giỏ hàng và fix hai bug còn lại trong ngày hôm nay trong khi anh chuẩn bị test report lúc 14 giờ và chị gửi danh sách case bổ sung trước 13 giờ."
    },
    {
        "transcript": "Chào sếp chúng ta cần xin duyệt gấp chi phí mua thêm 10 máy tính xách tay cho team dev mới tuyển. Chi phí ước tính 185 triệu đồng cho cấu hình cao giúp anh em code mượt mà và chạy nhiều môi trường song song. Thiết bị này sẽ giảm thời gian build từ 8 phút xuống còn 2 phút và tăng năng suất rõ rệt. Ok chi phí hợp lý vì team đang mở rộng dự án lớn. Anh tiến hành báo giá từ ba nhà cung cấp và mua ngay trong tuần này sau khi được duyệt. Chị sẽ hỗ trợ kiểm tra hợp đồng và gửi kế toán thanh toán trước ngày 25. Đồng ý tôi liên hệ nhà cung cấp và gửi báo giá chi tiết lúc 10 giờ sáng mai. Tôi test thử một máy demo để đảm bảo cấu hình phù hợp từ chiều nay. Tốt chúng ta không kéo dài làm luôn để kịp deadline. Cảm ơn sếp đã duyệt nhanh. Kết thúc hội ý nhanh ở đây.",
        "reference": "[DUYỆT CHI PHÍ THIẾT BỊ]:\nSếp đã duyệt chi phí mua 10 máy tính xách tay với tổng số tiền 185 triệu đồng cho team dev. Anh tiến hành báo giá và mua ngay trong tuần này đồng thời gửi hợp đồng cho kế toán thanh toán trước ngày 25 trong khi chị hỗ trợ kiểm tra và cập nhật thông tin thiết bị cho toàn team."
    },
    {
        "transcript": "Chào sếp và anh chị chúng ta chốt font chữ cho toàn bộ giao diện hôm nay. Tôi đề xuất font Roboto kết hợp Inter để trông hiện đại chuyên nghiệp và dễ đọc trên mọi thiết bị. Font này khớp hoàn toàn với brand guideline của khách hàng và rõ nét hơn so với font mặc định trước đó. Ok anh thấy Roboto rất phù hợp vì nó hỗ trợ tốt tiếng Việt và nhìn sang trọng. Sếp cũng đồng ý font này sẽ giúp giao diện đồng nhất hơn. Chúng ta chốt Roboto làm font chính thức. Chị áp dụng ngay vào toàn bộ file Figma bao gồm button text và heading. Anh review code CSS liên quan sau khi nhận file update. Đồng ý tôi sẽ chỉnh toàn bộ và gửi bản mới nhất cho mọi người. Tôi gửi file cuối cùng trước 15 giờ để kịp deadline deploy tuần này. Tốt không thay đổi gì nữa. Cảm ơn anh chị đã góp ý nhanh. Kết thúc hội ý ngắn gọn ở đây.",
        "reference": "[CHỐT FONT CHỮ THIẾT KẾ]:\nĐội ngũ thống nhất chọn font Roboto làm font chính thức cho toàn bộ giao diện sản phẩm. Chị áp dụng ngay vào file Figma và gửi bản update trước 15 giờ hôm nay để anh review code CSS liên quan."
    },
    {
        "transcript": "Chào anh chị chúng ta thông báo nhanh lịch team building quý 2 sắp tới. Công ty tổ chức team building hai ngày một đêm vào cuối tuần 17-18 tháng 5 tại resort Bình Dương. Toàn đội làm việc bình thường đến hết ngày 16 tháng 5 để hoàn tất task quan trọng. Sau đó di chuyển chiều thứ Sáu và quay lại làm việc vào sáng thứ Hai ngày 19 tháng 5. Anh gửi email thông báo chi tiết kèm form đăng ký cho mọi người hôm nay để thu thập danh sách tham gia. Chị cũng chuẩn bị nội dung hoạt động và gửi cho ban tổ chức trước khi nghỉ. Ok tôi soạn email ngay và gửi trước 11 giờ. Tôi hỗ trợ anh nếu cần thêm thông tin về chi phí và lịch trình. Đồng ý chúng ta giữ nhịp làm việc bình thường đến ngày 16. Tốt không có thay đổi gì khác. Cảm ơn anh chị đã lưu ý lịch team building. Kết thúc hội ý nhanh ở đây.",
        "reference": "[THÔNG BÁO LỊCH TEAM BUILDING]:\nCông ty tổ chức team building hai ngày một đêm ngày 17-18 tháng 5 tại resort Bình Dương. Toàn đội làm việc bình thường đến hết ngày 16 tháng 5 sau đó di chuyển và trở lại làm việc vào sáng ngày 19 tháng 5. Anh gửi email thông báo và form đăng ký cho toàn team trước ngày mai để đảm bảo thu thập danh sách kịp thời."
    },
    {
        "transcript": "Chào sếp và anh chị khách hàng đề nghị thay đổi yêu cầu tính năng chat hỗ trợ tuần này. Tôi gợi ý họp online thứ Tư lúc 9 giờ sáng để thảo luận chi tiết và thống nhất scope mới. Phần tài liệu yêu cầu cập nhật tôi sẽ chuẩn bị đầy đủ gửi cho mọi người xem trước ngày mai. Ok chị có thể demo prototype chat hiện tại trong cuộc họp để khách thấy rõ. Sếp tham gia 10 phút đầu để chào và xác nhận ngân sách bổ sung. Đồng ý 9 giờ sáng thứ Tư là phù hợp nhất. Anh confirm lịch với khách ngay hôm nay và cập nhật vào lịch chung cho team. Chị hỗ trợ trình bày phần kỹ thuật nếu khách hỏi thêm. Tôi chuẩn bị slide so sánh scope cũ và mới với dữ liệu ước tính. Tốt chúng ta sẽ có cuộc họp hiệu quả. Cảm ơn anh chị đã phối hợp nhanh. Kết thúc hội ý ngắn ở đây.",
        "reference": "[PHỐI HỢP THAY ĐỔI YÊU CẦU KHÁCH HÀNG]:\nĐội ngũ chốt lịch họp online với khách hàng vào 9 giờ sáng thứ Tư tuần này để thảo luận thay đổi tính năng chat hỗ trợ. Anh chuẩn bị slide so sánh scope chị hỗ trợ demo prototype và sếp tham gia 10 phút đầu để xác nhận ngân sách bổ sung."
    },

    # --- 10 MẪU TRUNG BÌNH ---
    {
        "transcript": "Chúng ta bắt đầu buổi brainstorming hôm nay để lên ý tưởng Marketing tháng tới trước khi deadline đề xuất cuối tuần. Tôi nghĩ chủ đề Hành trình xanh rất hợp vì sản phẩm của chúng ta đang đẩy mạnh mảng thân thiện môi trường và khách hàng hiện nay quan tâm nhiều đến bền vững. Ý tưởng này có thể triển khai thử thách tái chế trên mạng xã hội để người dùng chia sẻ ảnh trước sau và tag thương hiệu. Nhưng anh thì thấy chủ đề này hơi chung chung nên cần thêm yếu tố giải trí mới thu hút được giới trẻ. Chị lại góp ý nên kết hợp với influencer địa phương để tạo câu chuyện gần gũi hơn. Sau khi tranh luận một lúc chúng ta thấy Hành trình xanh vẫn là lựa chọn tốt nhất vì nó khớp với định hướng công ty năm nay. Ok quay lại vấn đề thứ hai là chọn kênh phân phối. TikTok đang hot với video ngắn nên chúng bản thân tiên nó nhưng Instagram vẫn giữ lượng follower ổn định. Tôi đề xuất phân bổ ngân sách 60 phần trăm cho TikTok để đẩy reach nhanh. Anh phản đối vì chi phí influencer trên TikTok cao hơn và sợ không đo lường được chuyển đổi. Chị thì đồng ý với Instagram vì story dễ tương tác hơn. Chúng ta tranh luận thêm về dữ liệu analytics tháng trước thấy TikTok mang lại engagement cao gấp đôi. Nhân tiện hôm nay trời nóng quá trưa nay ăn gì cho mát anh chị có gợi ý món nào không. Hôm qua tôi dẫn con đi ăn kem mà trời oi bức làm cả nhà mệt. Ừ nhưng mà quay lại công việc đi. Cuối cùng chúng ta chốt TikTok và Instagram là hai kênh chính với tỷ lệ 60 40 như vậy. Tôi sẽ soạn proposal chi tiết gửi sếp trước thứ sáu. Tốt lắm ý tưởng này khả thi và chúng ta thống nhất nhanh. Buổi họp kết thúc ở đây nhưng nhớ follow up email sau.",
        "reference": "[CHỦ ĐỀ CAMPAIGN]:\nĐội ngũ thống nhất chọn chủ đề campaign Marketing tháng tới là Hành trình xanh với trọng tâm quảng bá dòng sản phẩm thân thiện môi trường và khuyến khích khách hàng tham gia thử thách tái chế để tăng tương tác thương hiệu.\n[KÊNH PHÂN PHỐI]:\nToàn team chốt sử dụng TikTok và Instagram làm kênh chính đồng thời phân bổ ngân sách 60 phần trăm cho video ngắn và 40 phần trăm cho story tương tác nhằm tối ưu chi phí và tiếp cận đối tượng trẻ tuổi."
    },
    {
        "transcript": "Buổi đánh giá hai ứng viên phỏng vấn hôm nay chúng ta cần chốt quyết định nhanh để gửi offer trước cuối tuần. Trước tiên nói về ứng viên A vị trí Frontend. CV anh ấy đẹp kinh nghiệm ba năm và project cá nhân trên GitHub khá ấn tượng. Tôi thấy code React của anh ấy sạch và tối ưu nhưng lúc phỏng vấn trả lời hơi lúng túng về performance optimization. Chị lại nghĩ điểm yếu đó không sao vì anh ấy có thái độ học hỏi nhanh và team chúng ta đang cần người năng động. Anh thì lo ngại kinh nghiệm thực tế chưa đa dạng nhưng sau khi xem portfolio chung chúng ta thấy anh ấy phù hợp hơn so với yêu cầu. Sau tranh luận kỹ chúng ta thống nhất chọn ứng viên A. Bây giờ chuyển sang ứng viên B vị trí Marketing. CV chị ấy rất mạnh có bằng cấp cao nhưng khi hỏi về campaign thực tế thì trả lời khá mơ hồ. Tôi đề xuất loại vì kỹ năng phân tích dữ liệu chưa sâu trong khi quý này chúng ta cần người chạy ads hiệu quả. Chị phản đối vì chị ấy có ý tưởng sáng tạo và có thể đào tạo thêm. Anh thì đồng tình với tôi vì thời gian onboarding gấp. Chúng ta tranh luận thêm về kỹ năng mềm thấy ứng viên B hơi thiếu kinh nghiệm thực chiến. Nhân tiện trưa nay ăn gì anh chị. Tôi đang thèm phở nhưng trời mưa gió làm con tôi hôm qua bị ho cả đêm. Thôi quay lại đi. Cuối cùng chúng ta chốt loại ứng viên B để tìm người phù hợp hơn. Quyết định này hợp lý và team sẽ follow up offer cho ứng viên A ngay chiều nay. Buổi brainstorming kết thúc ở đây cảm ơn mọi người góp ý thẳng thắn.",
        "reference": "[ĐÁNH GIÁ ỨNG VIÊN A]:\nĐội ngũ thống nhất chọn ứng viên A cho vị trí Frontend Developer vì kinh nghiệm thực tế ba năm và kỹ năng React mạnh mẽ dù điểm phỏng vấn hơi thấp nhưng thái độ học hỏi tốt sẽ phù hợp với team hiện tại.\n[ĐÁNH GIÁ ỨNG VIÊN B]:\nToàn team chốt loại ứng viên B cho vị trí Marketing Specialist vì thiếu kinh nghiệm chạy campaign thực tế dù CV ấn tượng nhưng kỹ năng phân tích dữ liệu chưa đủ sâu để hỗ trợ mục tiêu quý này."
    },
    {
        "transcript": "Hôm nay chúng ta brainstorm chọn địa điểm và hoạt động cho Team Building quý này để tăng tinh thần đội ngũ sau giai đoạn bận rộn. Trước hết bàn về địa điểm. Resort ven biển Vũng Tàu tôi thấy hợp vì view đẹp và có hồ bơi. Chi phí chỉ khoảng hai mươi triệu cho hai mươi người. Anh thì thích núi Đà Lạt hơn vì không khí mát mẻ nhưng di chuyển xa hơn một ngày. Chị lại lo ngại thời tiết biển hay mưa bất chợt nên đề xuất homestay Đà Lạt. Chúng ta tranh luận khá lâu về chi phí và thời gian di chuyển thấy Vũng Tàu vẫn tiện lợi nhất cho mọi người. Ok sau khi cân nhắc kỹ chúng ta chốt resort Vũng Tàu. Nhân tiện hôm nay trời oi bức quá anh chị trưa ăn món gì mát lạnh không. Hôm qua vợ tôi nấu canh chua mà cả nhà khen ngon. Thôi quay lại hoạt động chính đi. Về hoạt động tôi đề xuất team challenge kết hợp workshop xây dựng lòng tin. Anh nghĩ thêm buổi lửa trại tối để mọi người chia sẻ câu chuyện cá nhân. Chị phản đối vì sợ lịch trình dày đặc làm mọi người mệt. Tôi thì thấy lửa trại sẽ giúp tăng gắn kết. Chúng ta tranh luận thêm về thời gian thấy lịch trình hai ngày một đêm vẫn thoải mái. Cuối cùng chốt hoạt động chính là team challenge workshop và lửa trại. Kế hoạch này hay và chúng ta sẽ book chỗ ngay tuần này. Buổi họp kết thúc nhưng nhớ update danh sách tham gia trước thứ tư.",
        "reference": "[ĐỊA ĐIỂM TỔ CHỨC]:\nĐội ngũ thống nhất chọn resort ven biển Vũng Tàu làm địa điểm Team Building vì chi phí hợp lý khoảng hai mươi triệu đồng và khoảng cách di chuyển chỉ ba giờ giúp toàn team thư giãn mà vẫn kết nối được.\n[HOẠT ĐỘNG CHÍNH]:\nToàn team chốt hoạt động chính là trò chơi team challenge kết hợp workshop xây dựng lòng tin và buổi tối lửa trại để tăng tinh thần đoàn kết trước khi về trong ngày chủ nhật."
    },
    {
        "transcript": "Buổi phân tích lỗi phần mềm hôm nay chúng ta tập trung vào hai lỗi chính đang ảnh hưởng khách hàng để chốt giải pháp nhanh. Trước tiên lỗi login người dùng báo không đăng nhập được dù mật khẩu đúng. Tôi nghĩ nguyên nhân là xung đột API authentication phiên bản cũ. Anh thì cho rằng do cache browser nhưng sau khi kiểm tra log chúng ta thấy rõ vấn đề từ server side. Chị góp ý update version mới sẽ an toàn hơn. Chúng ta tranh luận một lúc về rủi ro downtime nhưng cuối cùng thống nhất update ngay. Nhân tiện trưa nay anh chị ăn gì. Tôi đang thèm cơm tấm nhưng con tôi hôm nay đi học về kể chuyện bạn bè làm tôi nhớ hồi xưa. Thôi quay lại lỗi payment đi. Lỗi này khá nghiêm trọng vì khách hàng thanh toán bị từ chối giữa chừng. Tôi phân tích thấy do validation token hết hạn đột ngột. Anh đề xuất thêm retry mechanism để tự động thử lại. Chị lo ngại test chưa đủ nên cần thêm case edge. Chúng ta tranh luận kỹ về thời gian triển khai thấy fix trước thứ sáu là khả thi. Cuối cùng chốt kế hoạch rõ ràng. Quyết định này sẽ giúp hệ thống ổn định hơn và khách hàng hài lòng. Buổi brainstorming kết thúc ở đây chúng ta follow up ticket ngay.",
        "reference": "[PHÂN TÍCH LỖI LOGIN]:\nĐội ngũ thống nhất xác định nguyên nhân lỗi login là do xung đột API authentication cũ và quyết định update version mới trước cuối ngày hôm nay để khắc phục triệt để cho người dùng.\n[PHÂN TÍCH LỖI PAYMENT]:\nToàn team chốt lỗi payment xuất phát từ validation token hết hạn và lên kế hoạch fix bằng cách thêm retry mechanism cùng test case chi tiết trước khi deploy production vào thứ sáu."
    },
    {
        "transcript": "Chúng ta brainstorm lập kế hoạch đào tạo nội bộ quý này để nâng cao năng lực đội ngũ trước khi dự án mới bắt đầu. Trước hết bàn về đào tạo kỹ năng. Tôi đề xuất AI tool vì nó đang hot và giúp tiết kiệm thời gian viết code thiết kế. Anh thì thích soft skill hơn nhưng sau khi xem khảo sát nội bộ chúng ta thấy đa số muốn AI. Chị góp ý nên kết hợp cả hai nhưng ngân sách hạn chế nên ưu tiên AI trước. Chúng ta tranh luận khá lâu về hình thức trực tuyến hay offline thấy hai buổi online một buổi offline là cân bằng. Ok chốt kế hoạch đào tạo AI tool. Nhân tiện hôm nay trời mưa to anh chị có bị kẹt xe không. Hôm qua tôi đưa con đi học mà đường ngập nước làm muộn cả tiếng. Thôi quay lại kế hoạch đào tạo quy trình đi. Về quy trình họp hành hiện tại hơi lan man nên cần template agenda chuẩn. Tôi nghĩ thêm công cụ họp trực tuyến thống nhất sẽ giảm thời gian xuống dưới mười phút. Anh phản đối vì sợ mọi người không quen nhưng chị đồng ý vì nó giúp chốt công việc nhanh hơn. Chúng ta tranh luận thêm về chi tiết template thấy phiên bản đơn giản nhất khả thi. Cuối cùng chốt cập nhật quy trình mới. Kế hoạch này sẽ làm team hiệu quả hơn hẳn. Buổi họp kết thúc nhưng chúng ta gửi lịch đào tạo cho mọi người trước thứ hai.",
        "reference": "[KẾ HOẠCH ĐÀO TẠO KỸ NĂNG]:\nĐội ngũ thống nhất chọn đào tạo kỹ năng AI tool cho toàn team trong tháng tới với lịch hai buổi trực tuyến và một buổi thực hành offline để nâng cao hiệu suất công việc.\n[KẾ HOẠCH ĐÀO TẠO QUY TRÌNH]:\nToàn team chốt cập nhật quy trình họp hành mới bằng cách giới thiệu template agenda chuẩn và công cụ họp trực tuyến thống nhất nhằm giảm thời gian họp không cần thiết xuống dưới mười phút."
    },
    {
        "transcript": "Chúng ta bắt đầu buổi brainstorming hôm nay để lên ý tưởng tính năng mới cho ứng dụng mobile trước khi deadline đề xuất cuối tuần. Tôi nghĩ tính năng theo dõi sức khỏe hàng ngày rất hợp vì người dùng hiện nay quan tâm nhiều đến wellness và sản phẩm của chúng ta đang định hướng mảng lifestyle. Ý tưởng này có thể triển khai tính năng nhắc nhở uống nước, theo dõi bước chân và chia sẻ tiến độ với bạn bè. Nhưng anh thì thấy tính năng này hơi phổ biến nên cần thêm yếu tố gamification mới thu hút được giới trẻ. Chị lại góp ý nên tích hợp AI dự đoán cảm xúc để tạo trải nghiệm cá nhân hóa hơn. Sau khi tranh luận một lúc chúng ta thấy theo dõi sức khỏe vẫn là lựa chọn tốt nhất vì nó khớp với định hướng công ty năm nay. Ok quay lại vấn đề thứ hai là chọn công nghệ phát triển. React Native đang hot với cross-platform nên chúng ta ưu tiên nó nhưng Flutter vẫn giữ hiệu suất ổn định. Tôi đề xuất phân bổ 70 phần trăm ngân sách cho React Native để đẩy tiến độ nhanh. Anh phản đối vì đội ngũ hiện tại quen Flutter hơn và sợ thời gian học hỏi lâu. Chị thì đồng ý với React Native vì cộng đồng lớn và thư viện phong phú. Chúng ta tranh luận thêm về dữ liệu benchmark tháng trước thấy React Native giảm thời gian phát triển 40 phần trăm. Nhân tiện hôm nay trời mưa to quá trưa nay ăn gì cho ấm anh chị có gợi ý món nào không. Hôm qua tôi dẫn con đi ăn bún mà đường ngập nước làm cả nhà mệt. Ừ nhưng mà quay lại công việc đi. Cuối cùng chúng ta chốt React Native và Flutter là hai công nghệ chính với tỷ lệ 70 30 như vậy. Tôi sẽ soạn proposal chi tiết gửi sếp trước thứ sáu. Tốt lắm ý tưởng này khả thi và chúng ta thống nhất nhanh. Buổi họp kết thúc ở đây nhưng nhớ follow up email sau.",
        "reference": "[CHỦ ĐỀ TÍNH NĂNG MỚI]:\nĐội ngũ thống nhất chọn tính năng theo dõi sức khỏe hàng ngày cho ứng dụng mobile với trọng tâm gamification và AI dự đoán cảm xúc nhằm tăng tương tác người dùng và phù hợp định hướng lifestyle.\n[CÔNG NGHỆ PHÁT TRIỂN]:\nToàn team chốt sử dụng React Native làm công nghệ chính đồng thời kết hợp Flutter với tỷ lệ phân bổ ngân sách 70 phần trăm cho cross-platform và 30 phần trăm cho hiệu suất tối ưu."
    },
    {
        "transcript": "Buổi họp phân bổ ngân sách quý tới hôm nay chúng ta cần chốt quyết định nhanh để gửi đề xuất trước cuối tuần. Trước tiên nói về bộ phận Marketing. Tôi thấy ngân sách 450 triệu là hợp lý vì quý này cần đẩy mạnh campaign TikTok và influencer. Chị lại nghĩ nên tăng lên 550 triệu vì đối thủ đang chi mạnh. Anh thì lo ngại chi phí cao sẽ ảnh hưởng lợi nhuận nên đề xuất giữ nguyên 400 triệu và ưu tiên performance marketing. Sau tranh luận kỹ chúng ta thống nhất giữ 450 triệu cho Marketing. Bây giờ chuyển sang bộ phận Kỹ thuật. Ngân sách đề xuất 320 triệu cho server và tool mới. CV đội ngũ mạnh nhưng anh góp ý cần thêm 80 triệu cho AI tool. Chị phản đối vì thời gian triển khai gấp. Anh thì đồng tình giữ nguyên vì onboarding đội ngũ mới. Chúng ta tranh luận thêm về ROI thấy AI tool mang lại hiệu quả cao gấp 3 lần. Nhân tiện trưa nay ăn gì anh chị. Tôi đang thèm cà ri nhưng trời mưa gió làm con tôi hôm qua bị sổ mũi. Thôi quay lại đi. Cuối cùng chúng ta chốt ngân sách Kỹ thuật là 380 triệu sau khi cân nhắc. Quyết định này hợp lý và team sẽ follow up đề xuất gửi ban lãnh đạo ngay chiều nay. Buổi brainstorming kết thúc ở đây cảm ơn mọi người góp ý thẳng thắn.",
        "reference": "[PHÂN BỔ NGÂN SÁCH MARKETING]:\nĐội ngũ thống nhất phân bổ 450 triệu đồng cho bộ phận Marketing quý tới với trọng tâm campaign TikTok và performance marketing dù có ý kiến tăng ngân sách nhưng vẫn giữ mức hợp lý để tối ưu lợi nhuận.\n[PHÂN BỔ NGÂN SÁCH KỸ THUẬT]:\nToàn team chốt phân bổ 380 triệu đồng cho bộ phận Kỹ thuật bao gồm server mới và AI tool nhằm nâng cao hiệu suất phát triển trong khi vẫn kiểm soát chi phí chặt chẽ."
    },
    {
        "transcript": "Hôm nay chúng ta brainstorm lập kế hoạch content calendar cho quý 3 để tăng tương tác trên mạng xã hội. Trước hết bàn về chủ đề chính. Series video hướng dẫn sử dụng sản phẩm tôi thấy hợp vì khách hàng đang cần nội dung thực tế. Chi phí quay chỉ khoảng 120 triệu cho 12 video. Anh thì thích series story tương tác hơn vì engagement cao nhưng tốn thời gian chỉnh sửa. Chị lại lo ngại nội dung sản phẩm hơi khô nên đề xuất thêm behind the scene. Chúng ta tranh luận khá lâu về engagement data thấy video hướng dẫn vẫn mang lại like cao nhất. Ok sau khi cân nhắc kỹ chúng ta chốt series video hướng dẫn. Nhân tiện hôm nay trời oi bức quá anh chị trưa ăn món gì mát lạnh không. Hôm qua chồng tôi nấu canh chua mà cả nhà khen ngon. Thôi quay lại lịch đăng đi. Về lịch đăng tôi đề xuất 4 video mỗi tuần kết hợp 3 story hàng ngày. Anh nghĩ thêm live stream thứ Sáu để tương tác trực tiếp. Chị phản đối vì sợ lịch dày đặc làm team mệt. Tôi thì thấy live stream sẽ tăng gắn kết với khách. Chúng ta tranh luận thêm về thời gian thấy lịch 4 video một tuần vẫn thoải mái. Cuối cùng chốt lịch đăng chính là video hướng dẫn và live stream. Kế hoạch này hay và chúng ta sẽ triển khai ngay tuần này. Buổi họp kết thúc nhưng nhớ update content brief trước thứ tư.",
        "reference": "[CHỦ ĐỀ CONTENT QUÝ 3]:\nĐội ngũ thống nhất chọn series video hướng dẫn sử dụng sản phẩm làm chủ đề chính cho content calendar quý 3 vì engagement cao và phù hợp nhu cầu khách hàng thực tế.\n[LỊCH ĐĂNG VÀ ĐỊNH DẠNG]:\nToàn team chốt lịch đăng 4 video mỗi tuần kết hợp live stream thứ Sáu và 3 story hàng ngày nhằm tối ưu tương tác và duy trì sự hiện diện đều đặn trên mạng xã hội."
    },
    {
        "transcript": "Buổi đánh giá đề xuất từ nhà cung cấp hôm nay chúng ta tập trung vào hai gói dịch vụ chính để chốt nhà cung cấp cho quý tới. Trước tiên gói của nhà cung cấp A về hosting cloud. Tôi nghĩ giá 280 triệu/năm là cạnh tranh vì uptime 99.9 phần trăm và hỗ trợ 24/7. Anh thì cho rằng gói B của nhà cung cấp khác rẻ hơn 40 triệu nhưng tốc độ chậm hơn. Chị góp ý kiểm tra SLA kỹ vì downtime sẽ ảnh hưởng khách hàng. Chúng ta tranh luận một lúc về rủi ro nhưng cuối cùng thống nhất chọn gói A. Nhân tiện trưa nay anh chị ăn gì. Tôi đang thèm sushi nhưng con tôi hôm nay đi học về kể chuyện bạn bè làm tôi nhớ hồi xưa. Thôi quay lại gói logistics đi. Gói này khá quan trọng vì vận chuyển hàng hóa. Tôi phân tích thấy nhà cung cấp C có thời gian giao nhanh nhất chỉ 36 giờ. Anh đề xuất thêm hợp đồng dài hạn để giảm giá. Chị lo ngại chất lượng bao bì chưa đủ nên cần thêm case test. Chúng ta tranh luận kỹ về thời gian triển khai thấy ký hợp đồng trước thứ sáu là khả thi. Cuối cùng chốt kế hoạch rõ ràng. Quyết định này sẽ giúp chi phí vận hành ổn định hơn. Buổi brainstorming kết thúc ở đây chúng ta follow up hợp đồng ngay.",
        "reference": "[ĐÁNH GIÁ GÓI HOSTING]:\nĐội ngũ thống nhất chọn gói hosting cloud của nhà cung cấp A với giá 280 triệu đồng mỗi năm vì uptime cao và hỗ trợ 24/7 dù có đề xuất rẻ hơn nhưng ưu tiên chất lượng ổn định.\n[ĐÁNH GIÁ GÓI LOGISTICS]:\nToàn team chốt gói logistics của nhà cung cấp C với thời gian giao 36 giờ và ký hợp đồng dài hạn nhằm giảm chi phí đồng thời yêu cầu kiểm tra chất lượng bao bì kỹ trước khi triển khai."
    },
    {
        "transcript": "Chúng ta brainstorm lập kế hoạch phúc lợi nhân viên quý này để giữ chân nhân tài trước khi dự án mới bắt đầu. Trước hết bàn về chính sách thưởng. Tôi đề xuất tăng thưởng hiệu suất lên 15 phần trăm lương vì khảo sát nội bộ cho thấy mọi người mong muốn. Anh thì thích thưởng du lịch team hơn nhưng sau khi xem dữ liệu chúng ta thấy đa số ưu tiên tiền mặt. Chị góp ý nên kết hợp cả hai nhưng ngân sách hạn chế nên ưu tiên thưởng tiền trước. Chúng ta tranh luận khá lâu về hình thức thấy thưởng tiền mặt kết hợp du lịch một ngày là cân bằng. Ok chốt kế hoạch phúc lợi thưởng. Nhân tiện hôm nay trời nắng nóng anh chị có bị kẹt xe không. Hôm qua tôi đưa con đi học mà đường kẹt cứng làm muộn cả tiếng. Thôi quay lại chính sách nghỉ phép đi. Về nghỉ phép hiện tại hơi cứng nhắc nên cần linh hoạt hơn. Tôi nghĩ thêm 2 ngày phép năm cho nhân viên thâm niên trên 3 năm. Anh phản đối vì sợ ảnh hưởng tiến độ nhưng chị đồng ý vì giúp tăng sự hài lòng. Chúng ta tranh luận thêm về chi tiết thấy phiên bản linh hoạt nhất khả thi. Cuối cùng chốt cập nhật chính sách mới. Kế hoạch này sẽ làm team gắn bó hơn hẳn. Buổi họp kết thúc nhưng chúng ta gửi đề xuất phúc lợi cho ban lãnh đạo trước thứ hai.",
        "reference": "[KẾ HOẠCH PHÚC LỢI THƯỞNG]:\nĐội ngũ thống nhất chọn tăng thưởng hiệu suất lên 15 phần trăm lương kết hợp du lịch team một ngày cho toàn nhân viên trong quý này để khuyến khích hiệu quả công việc.\n[KẾ HOẠCH NGHỈ PHÉP]:\nToàn team chốt cập nhật chính sách nghỉ phép linh hoạt hơn bằng cách thêm 2 ngày phép năm cho nhân viên thâm niên trên 3 năm nhằm nâng cao sự hài lòng và giữ chân nhân tài."
    },

    # --- 10 MẪU DÀI ---
    {
        "transcript": "Chúng ta bắt đầu cuộc họp giao ban quý này với việc tổng kết doanh thu quý 4 trước khi đi sâu vào các vấn đề chiến lược lớn hơn. Số liệu chính thức cho thấy doanh thu chỉ đạt 85 phần trăm kế hoạch dẫn đến lỗ ròng lên tới 12 tỷ đồng so với quý trước tăng trưởng nhẹ 5 phần trăm. Nhưng một số ý kiến cho rằng con số này chưa phản ánh đúng thực tế vì báo cáo tài chính chưa cập nhật đầy đủ các khoản thu từ hợp đồng dài hạn ký cuối quý và một số khoản hoàn tiền chưa được trừ hết. Chị thì vặn vẹo mạnh mẽ rằng chính bộ phận sales đã thất bại hoàn toàn khi bỏ lỡ 35 hợp đồng lớn do thiếu follow up khách hàng và không nắm bắt được thay đổi nhu cầu thị trường. Anh lại phản bác rằng lỗi nằm ở marketing khi campaign quý 4 chi tiêu 18 tỷ nhưng chỉ mang về 7 phần trăm lead chất lượng và hoàn toàn không cạnh tranh nổi với đối thủ ABC đang chiếm 28 phần trăm thị phần bằng cách giảm giá sâu. Sau khi cãi vã gay gắt và nêu ra từng con số cụ thể trong bảng báo cáo chúng ta tạm chốt rằng tình hình tài chính rất nghiêm trọng và buộc phải cắt giảm nhân sự ngay lập tức. Ban đầu phương án A là cắt giảm 20 phần trăm nhân sự ở sales và marketing để tiết kiệm ngay 6 tỷ đồng mỗi quý. Nhưng anh phản đối kịch liệt vì mất nhân tài sẽ làm doanh số quý 1 sụt giảm thêm 15 phần trăm và đưa ra ví dụ dài dòng về đối thủ XYZ năm ngoái cắt 25 phần trăm nhưng doanh thu vẫn tăng nhờ đào tạo lại nhân viên. Chị thì đồng ý cắt nhưng chỉ ở mức 10 phần trăm và ưu tiên giữ người có kinh nghiệm trên 5 năm. Chúng ta tranh luận thêm về trách nhiệm của từng bộ phận và đổ lỗi cho nhau suốt gần nửa giờ. Nhân tiện thị trường hiện nay đang suy thoái mạnh với lạm phát cao và lãi suất ngân hàng tăng khiến khách hàng thắt chặt chi tiêu trong khi đối thủ cạnh tranh ra sản phẩm mới giá rẻ hơn 22 phần trăm. Sau khi chuyển sang bàn vấn đề điều chỉnh ngân sách marketing chúng ta quay lại và lật kèo hoàn toàn quyết định chỉ cắt giảm 15 phần trăm nhân sự thay vì 20 phần trăm ban đầu để tránh ảnh hưởng đến năng lực bán hàng. Tiếp tục với việc điều chỉnh ngân sách marketing ban đầu mọi người đồng ý giảm 40 phần trăm nhưng sau khi nghe phân tích ROI từ digital campaign thì lật kèo xuống còn 25 phần trăm và chuyển hết sang performance marketing. Chúng ta vặn vẹo từng khoản chi tiết trong bảng ngân sách cũ và đưa ra ví dụ về chiến dịch thất bại quý trước tốn 9 tỷ nhưng chỉ mang lại 120 đơn hàng. Chuyển sang tối ưu hóa vận hành thì có ý kiến cho rằng quy trình hiện tại quá chậm dẫn đến mất 18 phần trăm khách hàng do thời gian giao hàng dài. Anh nêu ví dụ cụ thể về đối thủ ABC áp dụng ERP và giảm thời gian xử lý từ 5 ngày xuống còn 1 ngày rưỡi. Sau tranh luận căng thẳng chúng ta chốt áp dụng phần mềm ERP mới ngay trong quý 1. Nhưng sau khi bàn kế hoạch quý 1 thì quay lại lật kèo một lần nữa về cắt giảm nhân sự vì lo ngại nếu cắt quá sâu sẽ không đạt mục tiêu doanh thu tăng 22 phần trăm. Cuối cùng sau nhiều giờ cãi vã và đổ lỗi lẫn nhau chúng ta thống nhất toàn bộ 5 vấn đề như đã chốt nhưng vẫn còn ý kiến bất đồng về việc ai chịu trách nhiệm chính thức theo dõi tiến độ. Thị trường tiếp tục khó khăn với đối thủ XYZ mở rộng nhà máy mới và dự kiến chiếm thêm 12 phần trăm thị phần trong năm nay nên chúng ta buộc phải hành động quyết liệt hơn nữa.",
        "reference": "[TÌNH HÌNH TÀI CHÍNH QUÝ 4]:\nĐội ngũ chốt xác nhận doanh thu quý 4 chỉ đạt 85 phần trăm kế hoạch do suy thoái kinh tế và cạnh tranh khốc liệt từ đối thủ dẫn đến lỗ ròng 12 tỷ đồng và buộc phải siết chặt chi phí vận hành toàn công ty ngay từ quý 1 năm sau.\n[CẮT GIẢM NHÂN SỰ]:\nToàn team quyết định cắt giảm 15 phần trăm nhân sự ở bộ phận sales và marketing để tiết kiệm 4 tỷ 8 trăm triệu đồng mỗi quý trong khi giữ nguyên lực lượng kỹ thuật cốt lõi và hỗ trợ đào tạo chuyển đổi vị trí cho nhân viên bị ảnh hưởng.\n[ĐIỀU CHỈNH NGÂN SÁCH MARKETING]:\nChốt giảm ngân sách marketing quý 1 xuống còn 25 phần trăm so với quý 4 và chuyển hướng hoàn toàn sang digital performance marketing nhằm tối ưu ROI thay vì các chiến dịch truyền thống tốn kém.\n[TỐI ƯU HÓA VẬN HÀNH]:\nĐội ngũ thống nhất tái cấu trúc quy trình nội bộ bằng cách áp dụng phần mềm ERP mới để giảm thời gian xử lý đơn hàng từ 72 giờ xuống dưới 36 giờ và cắt giảm chi phí logistics 18 phần trăm.\n[KẾ HOẠCH QUÝ 1]:\nToàn công ty chốt mục tiêu doanh thu quý 1 tăng 22 phần trăm so với quý 4 thông qua mở rộng kênh bán hàng trực tuyến và ký hợp đồng độc quyền với ba nhà phân phối lớn tại miền Nam."
    },
    {
        "transcript": "Cuộc họp chiến lược hôm nay tập trung vào xử lý khủng hoảng truyền thông sau khi lô sản phẩm bị khiếu nại hàng loạt trên mạng xã hội và báo chí. Tình hình rất nghiêm trọng khi có hơn 4500 bình luận tiêu cực chỉ trong 48 giờ qua và doanh số giảm đột ngột 35 phần trăm. Nhưng một số ý kiến cho rằng đây chỉ là tin đồn từ đối thủ cạnh tranh chứ không phải lỗi thực tế của sản phẩm. Chị thì vặn lại rằng chính bộ phận kiểm soát chất lượng đã bỏ qua báo cáo kiểm tra lô hàng trước khi xuất xưởng dẫn đến lỗi này. Anh lại đổ lỗi cho marketing vì không giám sát nội dung quảng cáo chặt chẽ khiến khách hàng kỳ vọng quá cao. Sau khi cãi vã gay gắt và nêu từng case khách hàng cụ thể chúng ta tạm chốt phải ra thông cáo xin lỗi ngay trong 24 giờ. Ban đầu phương án A là chỉ đăng trên website và fanpage nhưng sau khi tranh luận về mức độ lan tỏa thì lật kèo sang đăng trên tất cả kênh bao gồm báo chí và TikTok. Chúng ta vặn vẹo từng chữ trong bản nháp thông cáo và tranh cãi về việc có nên đề cập lỗi cụ thể hay không. Nhân tiện thị trường hiện nay đối thủ ABC đang khai thác triệt để sự cố này bằng cách chạy quảng cáo so sánh sản phẩm của họ an toàn hơn 100 phần trăm. Sau khi chuyển sang chiến lược truyền thông dài hạn chúng ta quay lại và lật kèo một lần nữa về thông cáo vì lo ngại nếu xin lỗi quá sớm sẽ làm giảm uy tín thương hiệu lâu dài. Tiếp tục bàn về hợp tác đối tác thì có ý kiến đề xuất ký ngay với hai công ty truyền thông lớn để kiểm soát tin đồn. Anh phản đối vì chi phí cao lên tới 8 tỷ nhưng chị đưa ra ví dụ đối thủ XYZ từng xử lý khủng hoảng tương tự và phục hồi doanh số trong 3 tuần nhờ hợp tác này. Sau tranh luận căng thẳng chúng ta chốt ký hợp đồng khẩn cấp. Nhưng sau khi bàn đến điều chỉnh sản phẩm thì quay lại lật kèo về chi phí hợp tác và quyết định chỉ ký với một công ty thay vì hai. Chúng ta tiếp tục đổ lỗi cho nhau về nguyên nhân gốc rễ của khủng hoảng và nêu ra hàng loạt ví dụ dài dòng từ các công ty khác trên thị trường. Thị trường truyền thông hiện nay thay đổi rất nhanh với thuật toán mạng xã hội ưu tiên nội dung tiêu cực nên chúng ta phải hành động quyết liệt hơn. Chuyển sang kế hoạch phục hồi doanh số ban đầu mọi người đồng ý giảm giá 20 phần trăm nhưng sau khi nghe phân tích thiệt hại từ khủng hoảng thì lật kèo lên 30 phần trăm trong hai tuần để đẩy mạnh doanh số ngay. Cuối cùng sau gần ba giờ cãi vã gay gắt và tranh luận từng con số thiệt hại chúng ta thống nhất toàn bộ năm vấn đề nhưng vẫn còn bất đồng về người chịu trách nhiệm theo dõi chỉ số truyền thông hàng ngày. Đối thủ đang tăng tốc chiếm thị phần nên nếu không xử lý triệt để thì năm nay chúng ta có nguy cơ mất thêm 18 phần trăm doanh thu.",
        "reference": "[PHẢN ỨNG KHỦNG HOẢNG]:\nĐội ngũ chốt ra thông cáo chính thức xin lỗi công khai trên tất cả kênh truyền thông trong vòng 24 giờ và cam kết thu hồi toàn bộ lô sản phẩm lỗi để lấy lại lòng tin khách hàng.\n[CHIẾN LƯỢC TRUYỀN THÔNG]:\nToàn team quyết định chuyển hướng sang chiến dịch kể chuyện khách hàng hài lòng kết hợp influencer uy tín nhằm bù đắp hình ảnh tiêu cực và tăng tương tác lên 40 phần trăm trong quý tới.\n[HỢP TÁC ĐỐI TÁC]:\nChốt ký hợp đồng khẩn cấp với hai công ty truyền thông lớn để kiểm soát tin đồn và đẩy nội dung tích cực thay vì để đối thủ khai thác.\n[ĐIỀU CHỈNH SẢN PHẨM]:\nĐội ngũ thống nhất nâng cấp quy trình kiểm soát chất lượng với kiểm tra ba lớp và công bố báo cáo minh bạch hàng tháng để tránh lặp lại khủng hoảng.\n[KẾ HOẠCH PHỤC HỒI DOANH SỐ]:\nToàn công ty chốt chương trình khuyến mãi đặc biệt giảm 30 phần trăm trong hai tuần tới nhằm đẩy mạnh doanh số và bù đắp thiệt hại từ khủng hoảng."
    },
    {
        "transcript": "Hội nghị chiến lược hôm nay bàn kế hoạch tung sản phẩm mới toàn quốc sau khi prototype đã hoàn thiện. Nghiên cứu thị trường ban đầu cho thấy nhu cầu cao ở phân khúc trung cấp nhưng chúng ta cần điều chỉnh tính năng để phù hợp khí hậu từng vùng. Nhưng anh cho rằng nghiên cứu chưa sâu vì mẫu khảo sát chỉ 1200 người và chưa đại diện cho nông thôn. Chị thì vặn rằng nếu không hoàn tất trước ngày 20 tháng này thì sẽ trễ lịch ra mắt. Sau cãi vã gay gắt về phương pháp nghiên cứu chúng ta tạm chốt hoàn tất trước ngày 20. Ban đầu phương án teaser chỉ trên Facebook nhưng sau khi bàn chiến lược ra mắt thì lật kèo sang TikTok và Instagram vì đối thủ ABC đang làm rất tốt trên nền tảng này với hơn 2 triệu view trong tuần đầu. Chúng ta tranh luận dài dòng về chi phí influencer và vặn vẹo từng con số dự toán. Nhân tiện thị trường sản phẩm mới hiện nay cạnh tranh cực kỳ khốc liệt với đối thủ XYZ ra mắt cùng phân khúc và chiếm 22 phần trăm thị phần chỉ trong hai tháng. Sau khi chuyển sang phân phối và logistics chúng ta quay lại lật kèo về ngày ra mắt vì lo ngại chuỗi cung ứng chưa sẵn sàng. Tiếp tục với ngân sách thì có ý kiến cho rằng 45 tỷ là quá cao nhưng chị đưa ra ví dụ đối thủ từng đầu tư 60 tỷ và thu hồi vốn trong quý đầu. Sau tranh luận căng thẳng chúng ta chốt giữ nguyên ngân sách và định giá 2 triệu 8. Nhưng sau khi bàn kế hoạch marketing sau ra mắt thì lật kèo một lần nữa về giá bán lên 3 triệu 2 vì sợ cạnh tranh giá rẻ. Chúng ta đổ lỗi cho nhau về chậm trễ trong khâu prototype và nêu ra hàng loạt ví dụ từ năm trước khi sản phẩm cũ ra mắt thất bại do phân phối yếu. Thị trường đang thay đổi nhanh với xu hướng tiêu dùng xanh nên sản phẩm mới phải nhấn mạnh yếu tố này. Cuối cùng sau nhiều vòng cãi vã và lật kèo liên tục chúng ta thống nhất toàn bộ năm vấn đề nhưng vẫn còn tranh cãi về trách nhiệm bộ phận nào chịu trách nhiệm chính cho chỉ số bán hàng tháng đầu.",
        "reference": "[NGHIÊN CỨU THỊ TRƯỜNG]:\nĐội ngũ chốt hoàn tất nghiên cứu thị trường chi tiết trước ngày 20 tháng này và tập trung vào ba phân khúc khách hàng chính tại miền Bắc miền Trung và miền Nam để điều chỉnh tính năng sản phẩm.\n[CHIẾN LƯỢC RA MẮT]:\nToàn team quyết định tung sản phẩm mới vào ngày 15 tháng 6 với chiến dịch teaser kéo dài hai tuần trên TikTok và Instagram nhằm tạo buzz trước khi mở bán chính thức toàn quốc.\n[PHÂN PHỐI VÀ LOGISTICS]:\nChốt hợp tác với năm nhà phân phối lớn để bao phủ 100 phần trăm tỉnh thành và xây dựng kho hàng khu vực để giảm thời gian giao hàng xuống dưới 48 giờ.\n[NGÂN SÁCH VÀ GIÁ BÁN]:\nĐội ngũ thống nhất phân bổ ngân sách 45 tỷ đồng cho giai đoạn ra mắt và định giá sản phẩm ở mức 2 triệu 8 trăm nghìn đồng để cạnh tranh trực tiếp với đối thủ.\n[KẾ HOẠCH MARKETING SAU RA MẮT]:\nToàn công ty chốt chạy chương trình khuyến mãi hậu mãi trong ba tháng đầu với quà tặng trị giá 500 nghìn đồng nhằm duy trì doanh số và thu thập feedback khách hàng."
    },
    {
        "transcript": "Cuộc họp giao ban hôm nay tập trung vào tái cấu trúc công ty để thích ứng với môi trường kinh doanh thay đổi nhanh chóng. Cơ cấu tổ chức cũ đã quá cồng kềnh dẫn đến quyết định chậm trễ và mất cơ hội thị trường. Nhưng anh cho rằng thay đổi lớn như vậy sẽ gây xáo trộn nội bộ và làm giảm năng suất ít nhất 20 phần trăm trong quý đầu. Chị thì phản bác rằng giữ nguyên mô hình cũ thì công ty sẽ tụt hậu so với đối thủ ABC đã tái cấu trúc thành công năm ngoái. Sau cãi vã gay gắt về rủi ro chúng ta tạm chốt thành lập ba ban chiến lược mới. Ban đầu phương án chuyển đổi số chỉ đầu tư 15 tỷ nhưng sau khi bàn nguồn nhân lực thì lật kèo lên 28 tỷ vì cần hệ thống mạnh hơn. Chúng ta vặn vẹo từng hạng mục chi phí và đổ lỗi cho bộ phận IT chậm triển khai các dự án trước. Nhân tiện thị trường công nghệ hiện nay đối thủ XYZ đang dẫn đầu với AI analytics giúp họ dự báo doanh số chính xác 85 phần trăm. Sau khi chuyển sang ngân sách tái cấu trúc chúng ta quay lại lật kèo về cơ cấu tổ chức vì lo ngại mất kiểm soát. Tiếp tục tranh luận về thời gian triển khai thì có ý kiến cho rằng trước 30 tháng 9 là khả thi nhưng anh đưa ra ví dụ dự án cũ kéo dài gấp đôi thời gian. Sau nhiều giờ cãi vã và nêu ví dụ dài dòng từ các công ty khác chúng ta chốt hoàn tất đúng hạn. Nhưng sau khi bàn nguồn nhân lực thì lật kèo một lần nữa về đầu tư ERP vì sợ chi phí vượt ngân sách. Cuối cùng sau tranh luận căng thẳng và đổ lỗi lẫn nhau chúng ta thống nhất toàn bộ năm vấn đề nhưng vẫn còn bất đồng về người đứng đầu ban mới.",
        "reference": "[CƠ CẤU TỔ CHỨC MỚI]:\nĐội ngũ chốt thành lập ba ban chiến lược trực thuộc CEO thay vì mô hình cũ để tăng tốc độ ra quyết định và giảm tầng lớp quản lý trung gian.\n[CHUYỂN ĐỔI SỐ]:\nToàn team quyết định đầu tư 28 tỷ đồng vào hệ thống ERP và AI analytics trong vòng sáu tháng để tối ưu hóa toàn bộ quy trình vận hành nội bộ.\n[NGUỒN NHÂN LỰC]:\nChốt chương trình đào tạo lại 40 phần trăm nhân viên hiện tại và tuyển dụng thêm 25 chuyên gia công nghệ cao để hỗ trợ chuyển đổi số.\n[NGÂN SÁCH TÁI CẤU TRÚC]:\nĐội ngũ thống nhất cắt giảm 15 phần trăm chi phí hành chính và tái phân bổ vào quỹ đổi mới sản phẩm để đảm bảo tăng trưởng bền vững.\n[THỜI GIAN TRIỂN KHAI]:\nToàn công ty chốt hoàn tất tái cấu trúc trước ngày 30 tháng 9 năm nay với các mốc kiểm tra hàng tháng để theo dõi tiến độ."
    },
    {
        "transcript": "Hội nghị chiến lược hôm nay bàn kế hoạch mở rộng thị trường quốc tế sau khi doanh thu nội địa tăng trưởng chậm lại. Thị trường Thái Lan và Indonesia có tiềm năng lớn với dân số trẻ và nhu cầu sản phẩm tương tự Việt Nam. Nhưng anh cho rằng chọn hai thị trường cùng lúc sẽ làm phân tán nguồn lực và rủi ro thất bại cao. Chị thì vặn lại rằng nếu chỉ làm một thị trường thì chậm so với đối thủ ABC đã vào Indonesia từ năm ngoái và đạt 18 phần trăm thị phần. Sau cãi vã gay gắt về thứ tự ưu tiên chúng ta tạm chốt cả hai. Ban đầu ngân sách chỉ 20 tỷ nhưng sau khi bàn chiến lược xây dựng thương hiệu thì lật kèo lên 35 tỷ vì cần agency quốc tế chuyên nghiệp. Chúng ta tranh luận dài dòng về chi phí dịch thuật và vặn vẹo từng khoản trong bảng dự toán. Nhân tiện thị trường quốc tế hiện nay biến động mạnh với tỷ giá ngoại tệ và quy định nhập khẩu mới của Indonesia khiến logistics phức tạp. Sau khi chuyển sang logistics chúng ta quay lại lật kèo về thị trường mục tiêu vì lo ngại rủi ro pháp lý. Tiếp tục với đánh giá rủi ro thì có ý kiến đề xuất nhóm giám sát riêng nhưng anh phản đối vì tăng chi phí quản lý. Chị đưa ra ví dụ đối thủ XYZ gặp vấn đề tỷ giá và mất 10 tỷ đồng năm ngoái. Sau tranh luận căng thẳng chúng ta chốt thành lập nhóm. Nhưng sau khi bàn toàn bộ thì lật kèo một lần nữa về ngân sách vì sợ vượt dự toán. Cuối cùng sau gần bốn giờ cãi vã đổ lỗi và nêu ví dụ dài dòng từ đối thủ chúng ta thống nhất toàn bộ năm vấn đề nhưng vẫn còn tranh cãi về trách nhiệm theo dõi tiến độ mở rộng. Thị trường quốc tế cạnh tranh khốc liệt nên nếu không hành động nhanh thì sẽ mất cơ hội lớn trong năm nay.",
        "reference": "[THỊ TRƯỜNG MỤC TIÊU]:\nĐội ngũ chốt chọn thị trường Thái Lan và Indonesia làm điểm đến đầu tiên với kế hoạch mở văn phòng đại diện trong quý 2 năm nay để thử nghiệm sản phẩm.\n[CHIẾN LƯỢC XÂY DỰNG THƯƠNG HIỆU]:\nToàn team quyết định hợp tác với ba agency quốc tế để xây dựng hình ảnh thương hiệu cao cấp và chạy campaign localized phù hợp văn hóa từng nước.\n[LOGISTICS VÀ PHÂN PHỐI]:\nChốt ký hợp đồng với đối tác logistics khu vực để giảm chi phí vận chuyển 25 phần trăm và đảm bảo giao hàng trong 7 ngày.\n[NGÂN SÁCH MỞ RỘNG]:\nĐội ngũ thống nhất phân bổ 35 tỷ đồng cho năm đầu và tập trung 60 phần trăm vào marketing địa phương để đạt doanh thu 40 tỷ từ thị trường mới.\n[ĐÁNH GIÁ RỦI RO]:\nToàn công ty chốt thành lập nhóm giám sát rủi ro tỷ giá và quy định pháp lý để điều chỉnh chiến lược linh hoạt nếu có biến động."
    },
    {
        "transcript": "Chúng ta bắt đầu cuộc họp giao ban quý này với việc tổng kết doanh số bán hàng quý 1 trước khi đi sâu vào các vấn đề chiến lược lớn hơn. Số liệu chính thức cho thấy doanh số chỉ đạt 82 phần trăm kế hoạch dẫn đến lỗ ròng lên tới 9 tỷ 5 trăm triệu đồng so với quý trước tăng trưởng nhẹ 3 phần trăm. Nhưng một số ý kiến cho rằng con số này chưa phản ánh đúng thực tế vì báo cáo chưa cập nhật đầy đủ các khoản hoàn trả từ đối tác và một số đơn hàng lớn ký cuối quý vẫn đang chờ thanh toán. Chị thì vặn vẹo mạnh mẽ rằng chính bộ phận bán lẻ đã thất bại hoàn toàn khi bỏ lỡ 28 cửa hàng đối tác do thiếu đàm phán hợp đồng và không nắm bắt được thay đổi hành vi khách hàng sau dịch. Anh lại phản bác rằng lỗi nằm ở chuỗi cung ứng khi campaign quý 1 chi tiêu 15 tỷ nhưng chỉ mang về 6 phần trăm đơn hàng chất lượng và hoàn toàn không cạnh tranh nổi với đối thủ ABC đang chiếm 31 phần trăm thị phần bằng cách giao hàng trong 24 giờ. Sau khi cãi vã gay gắt và nêu ra từng con số cụ thể trong bảng báo cáo chúng ta tạm chốt rằng tình hình kinh doanh rất nghiêm trọng và buộc phải cắt giảm chi phí ngay lập tức. Ban đầu phương án A là cắt giảm 18 phần trăm chi phí marketing để tiết kiệm ngay 5 tỷ đồng mỗi quý. Nhưng anh phản đối kịch liệt vì mất khách hàng sẽ làm doanh số quý 2 sụt giảm thêm 17 phần trăm và đưa ra ví dụ dài dòng về đối thủ XYZ năm ngoái cắt 22 phần trăm nhưng doanh thu vẫn tăng nhờ tối ưu kênh online. Chị thì đồng ý cắt nhưng chỉ ở mức 12 phần trăm và ưu tiên giữ ngân sách cho TikTok Shop. Chúng ta tranh luận thêm về trách nhiệm của từng bộ phận và đổ lỗi cho nhau suốt gần nửa giờ. Nhân tiện thị trường hiện nay đang suy thoái mạnh với lạm phát cao và lãi suất ngân hàng tăng khiến khách hàng thắt chặt chi tiêu trong khi đối thủ cạnh tranh ra chương trình trả góp 0 phần trăm. Sau khi chuyển sang bàn vấn đề tối ưu chuỗi cung ứng chúng ta quay lại và lật kèo hoàn toàn quyết định chỉ cắt giảm 14 phần trăm chi phí thay vì 18 phần trăm ban đầu để tránh ảnh hưởng đến doanh số. Tiếp tục với việc điều chỉnh ngân sách online ban đầu mọi người đồng ý giảm 35 phần trăm nhưng sau khi nghe phân tích ROI từ livestream thì lật kèo xuống còn 22 phần trăm và chuyển hết sang TikTok Shop. Chúng ta vặn vẹo từng khoản chi tiết trong bảng ngân sách cũ và đưa ra ví dụ về chiến dịch thất bại quý trước tốn 7 tỷ nhưng chỉ mang lại 85 đơn hàng. Chuyển sang mở rộng kênh bán lẻ thì có ý kiến cho rằng quy trình hiện tại quá chậm dẫn đến mất 15 phần trăm đối tác do thời gian ký hợp đồng dài. Anh nêu ví dụ cụ thể về đối thủ ABC áp dụng hệ thống CRM và giảm thời gian ký từ 14 ngày xuống còn 3 ngày. Sau tranh luận căng thẳng chúng ta chốt áp dụng CRM mới ngay trong quý 2. Nhưng sau khi bàn kế hoạch quý 2 thì quay lại lật kèo một lần nữa về cắt giảm chi phí vì lo ngại nếu cắt quá sâu sẽ không đạt mục tiêu doanh số tăng 25 phần trăm. Cuối cùng sau nhiều giờ cãi vã và đổ lỗi lẫn nhau chúng ta thống nhất toàn bộ 5 vấn đề như đã chốt nhưng vẫn còn ý kiến bất đồng về việc ai chịu trách nhiệm chính thức theo dõi tiến độ. Thị trường tiếp tục khó khăn với đối thủ XYZ mở thêm 15 cửa hàng mới và dự kiến chiếm thêm 14 phần trăm thị phần trong năm nay nên chúng ta buộc phải hành động quyết liệt hơn nữa.",
        "reference": "[TÌNH HÌNH DOANH SỐ QUÝ 1]:\nĐội ngũ chốt xác nhận doanh số quý 1 chỉ đạt 82 phần trăm kế hoạch do suy thoái kinh tế và cạnh tranh khốc liệt từ đối thủ dẫn đến lỗ ròng 9 tỷ 5 trăm triệu đồng và buộc phải siết chặt chi phí vận hành toàn công ty ngay từ quý 2 năm nay.\n[CẮT GIẢM CHI PHÍ]:\nToàn team quyết định cắt giảm 14 phần trăm chi phí marketing và vận hành để tiết kiệm 4 tỷ 2 trăm triệu đồng mỗi quý trong khi giữ nguyên lực lượng bán lẻ cốt lõi và hỗ trợ đào tạo chuyển đổi kênh online cho nhân viên bị ảnh hưởng.\n[ĐIỀU CHỈNH NGÂN SÁCH ONLINE]:\nChốt giảm ngân sách online quý 2 xuống còn 22 phần trăm so với quý 1 và chuyển hướng hoàn toàn sang TikTok Shop nhằm tối ưu ROI thay vì các chiến dịch truyền thống tốn kém.\n[TỐI ƯU HÓA CHUỖI CUNG ỨNG]:\nĐội ngũ thống nhất áp dụng hệ thống CRM mới để giảm thời gian ký hợp đồng từ 14 ngày xuống dưới 3 ngày và cắt giảm chi phí logistics 15 phần trăm.\n[KẾ HOẠCH QUÝ 2]:\nToàn công ty chốt mục tiêu doanh số quý 2 tăng 25 phần trăm so với quý 1 thông qua mở rộng TikTok Shop và ký hợp đồng độc quyền với mười đối tác bán lẻ lớn tại miền Bắc."
    },
    {
        "transcript": "Cuộc họp chiến lược hôm nay tập trung vào xử lý khủng hoảng pháp lý sau khi công ty bị kiện tập thể bởi nhóm khách hàng về chất lượng dịch vụ. Tình hình rất nghiêm trọng khi có hơn 3200 khiếu nại chỉ trong 72 giờ qua và cổ phiếu giảm đột ngột 28 phần trăm. Nhưng một số ý kiến cho rằng đây chỉ là hành động của đối thủ cạnh tranh chứ không phải lỗi thực tế của công ty. Chị thì vặn lại rằng chính bộ phận pháp lý đã bỏ qua kiểm tra hợp đồng trước khi ký dẫn đến vụ kiện này. Anh lại đổ lỗi cho bộ phận dịch vụ khách hàng vì không giám sát khiếu nại chặt chẽ khiến khách hàng bức xúc. Sau khi cãi vã gay gắt và nêu từng case cụ thể chúng ta tạm chốt phải ra thông cáo chính thức trong 12 giờ. Ban đầu phương án A là chỉ gửi email cho khách hàng nhưng sau khi tranh luận về mức độ lan tỏa thì lật kèo sang đăng trên tất cả kênh bao gồm báo chí và LinkedIn. Chúng ta vặn vẹo từng chữ trong bản nháp và tranh cãi về việc có nên thừa nhận lỗi hay không. Nhân tiện thị trường hiện nay đối thủ ABC đang khai thác triệt để vụ kiện này bằng cách chạy quảng cáo dịch vụ của họ tốt hơn 100 phần trăm. Sau khi chuyển sang chiến lược pháp lý dài hạn chúng ta quay lại và lật kèo một lần nữa về thông cáo vì lo ngại nếu thừa nhận quá sớm sẽ làm giảm uy tín lâu dài. Tiếp tục bàn về hợp tác luật sư thì có ý kiến đề xuất ký ngay với hai công ty luật lớn để kiểm soát vụ kiện. Anh phản đối vì chi phí cao lên tới 6 tỷ nhưng chị đưa ra ví dụ đối thủ XYZ từng xử lý vụ kiện tương tự và thắng kiện trong 4 tuần nhờ hợp tác này. Sau tranh luận căng thẳng chúng ta chốt ký hợp đồng khẩn cấp. Nhưng sau khi bàn đến bồi thường thì quay lại lật kèo về chi phí và quyết định chỉ ký với một công ty thay vì hai. Chúng ta tiếp tục đổ lỗi cho nhau về nguyên nhân gốc rễ và nêu ra hàng loạt ví dụ dài dòng từ các công ty khác. Thị trường pháp lý hiện nay thay đổi rất nhanh với luật mới về bảo vệ người tiêu dùng nên chúng ta phải hành động quyết liệt hơn. Chuyển sang kế hoạch phục hồi uy tín ban đầu mọi người đồng ý bồi thường 15 phần trăm nhưng sau khi nghe phân tích thiệt hại thì lật kèo lên 25 phần trăm trong một tháng để giữ chân khách hàng. Cuối cùng sau gần ba giờ cãi vã gay gắt và tranh luận từng con số thiệt hại chúng ta thống nhất toàn bộ năm vấn đề nhưng vẫn còn bất đồng về người chịu trách nhiệm theo dõi tiến độ pháp lý hàng ngày. Đối thủ đang tăng tốc chiếm thị phần nên nếu không xử lý triệt để thì năm nay chúng ta có nguy cơ mất thêm 15 phần trăm doanh thu.",
        "reference": "[PHẢN ỨNG KHỦNG HOẢNG PHÁP LÝ]:\nĐội ngũ chốt ra thông cáo chính thức trong vòng 12 giờ và cam kết bồi thường cho toàn bộ khách hàng bị ảnh hưởng để lấy lại lòng tin.\n[CHIẾN LƯỢC PHÁP LÝ]:\nToàn team quyết định chuyển hướng sang hợp tác với luật sư chuyên nghiệp nhằm kiểm soát vụ kiện và đẩy mạnh truyền thông tích cực tăng tương tác lên 35 phần trăm trong quý tới.\n[HỢP TÁC LUẬT SƯ]:\nChốt ký hợp đồng khẩn cấp với hai công ty luật lớn để xử lý tập thể và tránh để đối thủ khai thác.\n[ĐIỀU CHỈNH DỊCH VỤ]:\nĐội ngũ thống nhất nâng cấp quy trình dịch vụ khách hàng với kiểm tra hai lớp và công bố báo cáo minh bạch hàng tháng để tránh lặp lại khủng hoảng.\n[KẾ HOẠCH PHỤC HỒI UY TÍN]:\nToàn công ty chốt chương trình bồi thường đặc biệt 25 phần trăm trong một tháng tới nhằm đẩy mạnh giữ chân khách hàng và bù đắp thiệt hại từ vụ kiện."
    },
    {
        "transcript": "Hội nghị chiến lược hôm nay bàn kế hoạch ra mắt nền tảng thương mại điện tử mới sau khi prototype đã hoàn thiện. Nghiên cứu thị trường ban đầu cho thấy nhu cầu cao ở phân khúc Gen Z nhưng chúng ta cần điều chỉnh giao diện để phù hợp xu hướng mobile-first. Nhưng anh cho rằng nghiên cứu chưa sâu vì mẫu khảo sát chỉ 1800 người và chưa đại diện cho khách hàng nông thôn. Chị thì vặn rằng nếu không hoàn tất trước ngày 15 tháng này thì sẽ trễ lịch ra mắt. Sau cãi vã gay gắt về phương pháp nghiên cứu chúng ta tạm chốt hoàn tất trước ngày 15. Ban đầu phương án teaser chỉ trên Facebook nhưng sau khi bàn chiến lược ra mắt thì lật kèo sang TikTok và Shopee vì đối thủ ABC đang làm rất tốt trên nền tảng này với hơn 3 triệu view trong tuần đầu. Chúng ta tranh luận dài dòng về chi phí influencer và vặn vẹo từng con số dự toán. Nhân tiện thị trường thương mại điện tử hiện nay cạnh tranh cực kỳ khốc liệt với đối thủ XYZ ra mắt cùng phân khúc và chiếm 26 phần trăm thị phần chỉ trong hai tháng. Sau khi chuyển sang tích hợp thanh toán chúng ta quay lại lật kèo về ngày ra mắt vì lo ngại hệ thống chưa sẵn sàng. Tiếp tục với ngân sách thì có ý kiến cho rằng 38 tỷ là quá cao nhưng chị đưa ra ví dụ đối thủ từng đầu tư 55 tỷ và thu hồi vốn trong quý đầu. Sau tranh luận căng thẳng chúng ta chốt giữ nguyên ngân sách và định giá gói premium 990 nghìn đồng. Nhưng sau khi bàn kế hoạch marketing sau ra mắt thì lật kèo một lần nữa về giá bán lên 1 triệu 2 vì sợ cạnh tranh giá rẻ. Chúng ta đổ lỗi cho nhau về chậm trễ trong khâu prototype và nêu ra hàng loạt ví dụ từ năm trước khi nền tảng cũ ra mắt thất bại do tích hợp yếu. Thị trường đang thay đổi nhanh với xu hướng mua sắm xanh nên nền tảng mới phải nhấn mạnh yếu tố này. Cuối cùng sau nhiều vòng cãi vã và lật kèo liên tục chúng ta thống nhất toàn bộ năm vấn đề nhưng vẫn còn tranh cãi về trách nhiệm bộ phận nào chịu trách nhiệm chính cho chỉ số chuyển đổi tháng đầu.",
        "reference": "[NGHIÊN CỨU THỊ TRƯỜNG]:\nĐội ngũ chốt hoàn tất nghiên cứu thị trường chi tiết trước ngày 15 tháng này và tập trung vào ba phân khúc khách hàng chính Gen Z Gen Y và khách hàng nông thôn để điều chỉnh giao diện mobile-first.\n[CHIẾN LƯỢC RA MẮT]:\nToàn team quyết định ra mắt nền tảng vào ngày 10 tháng 7 với chiến dịch teaser kéo dài hai tuần trên TikTok và Shopee nhằm tạo buzz trước khi mở bán chính thức toàn quốc.\n[TÍCH HỢP THANH TOÁN]:\nChốt hợp tác với bốn cổng thanh toán lớn để bao phủ 100 phần trăm giao dịch và xây dựng hệ thống server khu vực để giảm thời gian load dưới 2 giây.\n[NGÂN SÁCH VÀ GIÁ BÁN]:\nĐội ngũ thống nhất phân bổ ngân sách 38 tỷ đồng cho giai đoạn ra mắt và định giá gói premium ở mức 990 nghìn đồng để cạnh tranh trực tiếp với đối thủ.\n[KẾ HOẠCH MARKETING SAU RA MẮT]:\nToàn công ty chốt chạy chương trình khuyến mãi hậu mãi trong ba tháng đầu với voucher trị giá 300 nghìn đồng nhằm duy trì doanh số và thu thập feedback người dùng."
    },
    {
        "transcript": "Cuộc họp giao ban hôm nay tập trung vào tái cấu trúc bộ phận bán hàng để thích ứng với môi trường kinh doanh thay đổi nhanh chóng. Cơ cấu tổ chức cũ đã quá cồng kềnh dẫn đến quyết định chậm trễ và mất cơ hội thị trường. Nhưng anh cho rằng thay đổi lớn như vậy sẽ gây xáo trộn nội bộ và làm giảm năng suất ít nhất 22 phần trăm trong quý đầu. Chị thì phản bác rằng giữ nguyên mô hình cũ thì công ty sẽ tụt hậu so với đối thủ ABC đã tái cấu trúc thành công năm ngoái. Sau cãi vã gay gắt về rủi ro chúng ta tạm chốt thành lập bốn nhóm bán hàng chuyên biệt mới. Ban đầu phương án chuyển đổi số chỉ đầu tư 12 tỷ nhưng sau khi bàn nguồn nhân lực thì lật kèo lên 26 tỷ vì cần hệ thống CRM mạnh hơn. Chúng ta vặn vẹo từng hạng mục chi phí và đổ lỗi cho bộ phận IT chậm triển khai các dự án trước. Nhân tiện thị trường bán hàng hiện nay đối thủ XYZ đang dẫn đầu với AI dự báo nhu cầu giúp họ tăng doanh số chính xác 78 phần trăm. Sau khi chuyển sang ngân sách tái cấu trúc chúng ta quay lại lật kèo về cơ cấu tổ chức vì lo ngại mất kiểm soát. Tiếp tục tranh luận về thời gian triển khai thì có ý kiến cho rằng trước 25 tháng 8 là khả thi nhưng anh đưa ra ví dụ dự án cũ kéo dài gấp đôi thời gian. Sau nhiều giờ cãi vã và nêu ví dụ dài dòng từ các công ty khác chúng ta chốt hoàn tất đúng hạn. Nhưng sau khi bàn nguồn nhân lực thì lật kèo một lần nữa về đầu tư AI vì sợ chi phí vượt ngân sách. Cuối cùng sau tranh luận căng thẳng và đổ lỗi lẫn nhau chúng ta thống nhất toàn bộ năm vấn đề nhưng vẫn còn bất đồng về người đứng đầu nhóm bán hàng mới.",
        "reference": "[CƠ CẤU BỘ PHẬN BÁN HÀNG MỚI]:\nĐội ngũ chốt thành lập bốn nhóm bán hàng chuyên biệt trực thuộc CEO thay vì mô hình cũ để tăng tốc độ ra quyết định và giảm tầng lớp quản lý trung gian.\n[CHUYỂN ĐỔI SỐ BÁN HÀNG]:\nToàn team quyết định đầu tư 26 tỷ đồng vào hệ thống CRM và AI dự báo nhu cầu trong vòng năm tháng để tối ưu hóa toàn bộ quy trình bán hàng nội bộ.\n[NGUỒN NHÂN LỰC]:\nChốt chương trình đào tạo lại 35 phần trăm nhân viên bán hàng hiện tại và tuyển dụng thêm 18 chuyên gia sales công nghệ cao để hỗ trợ chuyển đổi số.\n[NGÂN SÁCH TÁI CẤU TRÚC]:\nĐội ngũ thống nhất cắt giảm 18 phần trăm chi phí hành chính bán hàng và tái phân bổ vào quỹ đào tạo và công cụ mới để đảm bảo tăng trưởng bền vững.\n[THỜI GIAN TRIỂN KHAI]:\nToàn công ty chốt hoàn tất tái cấu trúc bộ phận bán hàng trước ngày 25 tháng 8 năm nay với các mốc kiểm tra hàng tuần để theo dõi tiến độ."
    },
    {
        "transcript": "Hội nghị chiến lược hôm nay bàn kế hoạch hợp tác chiến lược với đối tác nước ngoài sau khi doanh thu nội địa tăng trưởng chậm lại. Thị trường Nhật Bản và Hàn Quốc có tiềm năng lớn với công nghệ cao và nhu cầu sản phẩm tương tự Việt Nam. Nhưng anh cho rằng chọn hai đối tác cùng lúc sẽ làm phân tán nguồn lực và rủi ro thất bại cao. Chị thì vặn lại rằng nếu chỉ làm một đối tác thì chậm so với đối thủ ABC đã hợp tác với Hàn Quốc từ năm ngoái và đạt 21 phần trăm thị phần. Sau cãi vã gay gắt về thứ tự ưu tiên chúng ta tạm chốt cả hai. Ban đầu ngân sách chỉ 18 tỷ nhưng sau khi bàn chiến lược đàm phán thì lật kèo lên 32 tỷ vì cần luật sư quốc tế chuyên nghiệp. Chúng ta tranh luận dài dòng về chi phí dịch thuật và vặn vẹo từng khoản trong bảng dự toán. Nhân tiện thị trường quốc tế hiện nay biến động mạnh với tỷ giá ngoại tệ và quy định xuất khẩu mới của Hàn Quốc khiến đàm phán phức tạp. Sau khi chuyển sang đàm phán hợp đồng chúng ta quay lại lật kèo về đối tác mục tiêu vì lo ngại rủi ro sở hữu trí tuệ. Tiếp tục với đánh giá rủi ro thì có ý kiến đề xuất nhóm giám sát riêng nhưng anh phản đối vì tăng chi phí quản lý. Chị đưa ra ví dụ đối thủ XYZ gặp vấn đề bản quyền và mất 8 tỷ đồng năm ngoái. Sau tranh luận căng thẳng chúng ta chốt thành lập nhóm. Nhưng sau khi bàn toàn bộ thì lật kèo một lần nữa về ngân sách vì sợ vượt dự toán. Cuối cùng sau gần bốn giờ cãi vã đổ lỗi và nêu ví dụ dài dòng từ đối thủ chúng ta thống nhất toàn bộ năm vấn đề nhưng vẫn còn tranh cãi về trách nhiệm theo dõi tiến độ hợp tác. Thị trường quốc tế cạnh tranh khốc liệt nên nếu không hành động nhanh thì sẽ mất cơ hội lớn trong năm nay.",
        "reference": "[ĐỐI TÁC MỤC TIÊU]:\nĐội ngũ chốt chọn đối tác Nhật Bản và Hàn Quốc làm điểm đến đầu tiên với kế hoạch ký biên bản ghi nhớ trong quý 3 năm nay để thử nghiệm hợp tác công nghệ.\n[CHIẾN LƯỢC ĐÀM PHÁN]:\nToàn team quyết định hợp tác với hai công ty luật quốc tế để xây dựng hợp đồng và chạy campaign chung phù hợp văn hóa từng nước.\n[HỢP ĐỒNG VÀ PHÂN PHỐI]:\nChốt ký hợp đồng với đối tác logistics khu vực để giảm chi phí vận chuyển 20 phần trăm và đảm bảo giao công nghệ trong 10 ngày.\n[NGÂN SÁCH HỢP TÁC]:\nĐội ngũ thống nhất phân bổ 32 tỷ đồng cho năm đầu và tập trung 55 phần trăm vào marketing chung để đạt doanh thu 45 tỷ từ thị trường mới.\n[ĐÁNH GIÁ RỦI RO]:\nToàn công ty chốt thành lập nhóm giám sát rủi ro sở hữu trí tuệ và tỷ giá để điều chỉnh chiến lược linh hoạt nếu có biến động."
    }
]

In [ ]:
import time
import re
import numpy as np
from rouge_score import rouge_scorer

# Hàm tự động chuyển đổi Bản chuẩn (có nhãn) về dạng Gạch đầu dòng phẳng
def convert_to_flat_reference(text):
    # Xóa các thẻ nhãn như [TÌNH HÌNH TÀI CHÍNH QUÝ 4]:
    clean_text = re.sub(r'\[.*?\]:', '', text)
    # Tách thành các dòng, xóa khoảng trắng thừa và thêm dấu gạch đầu dòng
    lines = [f"- {line.strip()}" for line in clean_text.split('\n') if line.strip()]
    return '\n'.join(lines)

def evaluate_summarization_pass1(summarizer, dataset):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

    results = {
        "Base_Qwen": {"r1": [], "r2": [], "rl": [], "time": []},
        "TextRank": {"r1": [], "r2": [], "rl": [], "time": []},
        "Chunking": {"r1": [], "r2": [], "rl": [], "time": []}
    }

    print("BẮT ĐẦU ĐÁNH GIÁ ROUGE (CHỈ Ở BƯỚC 1: TRÍCH XUẤT SỰ KIỆN PHẲNG)...\n" + "="*60)

    for i, data in enumerate(dataset):
        print(f"Đang xử lý Mẫu số {i+1}...")
        text = summarizer.clean_text(data["transcript"])

        # Tự động biến đổi Reference thành gạch đầu dòng
        flat_reference = convert_to_flat_reference(data["reference"])

        # --- QWEN CƠ BẢN ---
        start_time = time.time()
        base_pred = summarizer.generate_summary(text, is_final=True)
        results["Base_Qwen"]["time"].append(time.time() - start_time)
        scores = scorer.score(flat_reference, base_pred)
        results["Base_Qwen"]["r1"].append(scores['rouge1'].fmeasure)
        results["Base_Qwen"]["r2"].append(scores['rouge2'].fmeasure)
        results["Base_Qwen"]["rl"].append(scores['rougeL'].fmeasure)

        # --- TEXTRANK + QWEN ---
        start_time = time.time()
        tr_pred = summarizer.enhanced_textrank(text)
        results["TextRank"]["time"].append(time.time() - start_time)
        scores = scorer.score(flat_reference, tr_pred)
        results["TextRank"]["r1"].append(scores['rouge1'].fmeasure)
        results["TextRank"]["r2"].append(scores['rouge2'].fmeasure)
        results["TextRank"]["rl"].append(scores['rougeL'].fmeasure)

        # --- CHUNKING + QWEN ---
        start_time = time.time()
        chunk_pred = summarizer.chunking_method(text)
        results["Chunking"]["time"].append(time.time() - start_time)
        scores = scorer.score(flat_reference, chunk_pred)
        results["Chunking"]["r1"].append(scores['rouge1'].fmeasure)
        results["Chunking"]["r2"].append(scores['rouge2'].fmeasure)
        results["Chunking"]["rl"].append(scores['rougeL'].fmeasure)

    # --- IN KẾT QUẢ TỔNG HỢP ---
    print("\n" + "="*60)
    print("KẾT QUẢ ĐÁNH GIÁ TRUNG BÌNH BƯỚC 1 (Dành cho Bảng 3.1)")
    print("="*60)
    for method, metrics in results.items():
        print(f"--- {method} ---")
        print(f"ROUGE-1: {np.mean(metrics['r1'])*100:.2f}% | ROUGE-2: {np.mean(metrics['r2'])*100:.2f}% | ROUGE-L: {np.mean(metrics['rl'])*100:.2f}%")
        print(f"Thời gian Tóm tắt: {np.mean(metrics['time']):.2f} giây\n")

# KÍCH HOẠT
evaluate_summarization_pass1(summarizer, test_data)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BẮT ĐẦU ĐÁNH GIÁ ROUGE (CHỈ Ở BƯỚC 1: TRÍCH XUẤT SỰ KIỆN PHẲNG)...
Đang xử lý Mẫu số 1...
Đang xử lý Mẫu số 2...
Đang xử lý Mẫu số 3...
Đang xử lý Mẫu số 4...
Đang xử lý Mẫu số 5...
Đang xử lý Mẫu số 6...
Đang xử lý Mẫu số 7...
Đang xử lý Mẫu số 8...
Đang xử lý Mẫu số 9...
Đang xử lý Mẫu số 10...
Đang xử lý Mẫu số 11...
Đang xử lý Mẫu số 12...
Đang xử lý Mẫu số 13...
Đang xử lý Mẫu số 14...
Đang xử lý Mẫu số 15...
Đang xử lý Mẫu số 16...
Đang xử lý Mẫu số 17...
Đang xử lý Mẫu số 18...
Đang xử lý Mẫu số 19...
Đang xử lý Mẫu số 20...
Đang xử lý Mẫu số 21...
Đang xử lý Mẫu số 22...
Đang xử lý Mẫu số 23...
Đang xử lý Mẫu số 24...
Đang xử lý Mẫu số 25...
Đang xử lý Mẫu số 26...
Đang xử lý Mẫu số 27...
Đang xử lý Mẫu số 28...
Đang xử lý Mẫu số 29...
Đang xử lý Mẫu số 30...

KẾT QUẢ ĐÁNH GIÁ TRUNG BÌNH BƯỚC 1 (Dành cho Bảng 3.1)
--- Base_Qwen ---
ROUGE-1: 63.61% | ROUGE-2: 36.40% | ROUGE-L: 38.02%
Thời gian Tóm tắt: 12.99 giây

--- TextRank ---
ROUGE-1: 66.64% | ROUGE-2: 33.55% | ROUGE-L: 38.

In [ ]:
import time
import re
import numpy as np
from rouge_score import rouge_scorer

# Hàm tự động chuyển đổi Bản chuẩn (có nhãn) về dạng Gạch đầu dòng phẳng
def convert_to_flat_reference(text):
    # Xóa các thẻ nhãn như [TÌNH HÌNH TÀI CHÍNH QUÝ 4]:
    clean_text = re.sub(r'\[.*?\]:', '', text)
    # Tách thành các dòng, xóa khoảng trắng thừa và thêm dấu gạch đầu dòng
    lines = [f"- {line.strip()}" for line in clean_text.split('\n') if line.strip()]
    return '\n'.join(lines)

def evaluate_summarization_pass1_grouped(summarizer, dataset):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

    # 1. Định nghĩa các nhóm và cấu trúc lưu kết quả
    groups = ["Ngắn", "Trung bình", "Dài"]
    methods = ["Base_Qwen", "TextRank", "Chunking"]

    # Khởi tạo dictionary lồng nhau: Nhóm -> Mô hình -> Các chỉ số
    results = {
        group: {
            method: {"r1": [], "r2": [], "rl": [], "time": []}
            for method in methods
        }
        for group in groups
    }

    print("BẮT ĐẦU ĐÁNH GIÁ ROUGE THEO NHÓM ĐỘ DÀI VĂN BẢN...\n" + "="*70)

    for i, data in enumerate(dataset):
        # CÁCH 1: Phân loại theo Index (Giả sử danh sách đã được sắp xếp sẵn: 10 - 10 - 10)
        if i < 10:
            current_group = "Ngắn"
        elif i < 20:
            current_group = "Trung bình"
        else:
            current_group = "Dài"

        # CÁCH 2: Phân loại tự động theo số từ (Bỏ comment để dùng nếu dataset bị trộn lộn xộn)
        # word_count = len(data["transcript"].split())
        # if word_count < 500: current_group = "Ngắn"           # Ngưỡng tự do bạn chỉnh
        # elif word_count < 1500: current_group = "Trung bình"
        # else: current_group = "Dài"

        print(f"Đang xử lý Mẫu số {i+1} (Phân loại: {current_group})...")
        text = summarizer.clean_text(data["transcript"])
        flat_reference = convert_to_flat_reference(data["reference"])

        # --- Base_Qwen ---
        start_time = time.time()
        base_pred = summarizer.generate_summary(text, is_final=True)
        results[current_group]["Base_Qwen"]["time"].append(time.time() - start_time)
        scores = scorer.score(flat_reference, base_pred)
        results[current_group]["Base_Qwen"]["r1"].append(scores['rouge1'].fmeasure)
        results[current_group]["Base_Qwen"]["r2"].append(scores['rouge2'].fmeasure)
        results[current_group]["Base_Qwen"]["rl"].append(scores['rougeL'].fmeasure)

        # --- TextRank ---
        start_time = time.time()
        tr_pred = summarizer.enhanced_textrank(text)
        results[current_group]["TextRank"]["time"].append(time.time() - start_time)
        scores = scorer.score(flat_reference, tr_pred)
        results[current_group]["TextRank"]["r1"].append(scores['rouge1'].fmeasure)
        results[current_group]["TextRank"]["r2"].append(scores['rouge2'].fmeasure)
        results[current_group]["TextRank"]["rl"].append(scores['rougeL'].fmeasure)

        # --- Chunking ---
        start_time = time.time()
        chunk_pred = summarizer.chunking_method(text)
        results[current_group]["Chunking"]["time"].append(time.time() - start_time)
        scores = scorer.score(flat_reference, chunk_pred)
        results[current_group]["Chunking"]["r1"].append(scores['rouge1'].fmeasure)
        results[current_group]["Chunking"]["r2"].append(scores['rouge2'].fmeasure)
        results[current_group]["Chunking"]["rl"].append(scores['rougeL'].fmeasure)

    # --- 2. IN KẾT QUẢ TỔNG HỢP THEO TỪNG NHÓM ---
    print("\n" + "="*80)
    print(f"{'KẾT QUẢ ĐÁNH GIÁ TRUNG BÌNH THEO NHÓM ĐỘ DÀI VĂN BẢN':^80}")
    print("="*80)

    for group in groups:
        # Kiểm tra xem nhóm này có mẫu nào không để tránh lỗi chia cho 0
        sample_count = len(results[group]["Base_Qwen"]["r1"])
        print(f"\n▶ NHÓM VĂN BẢN: [{group.upper()}] (Số lượng: {sample_count} mẫu)")
        print("-" * 80)
        print(f"{'Mô hình':<15} | {'ROUGE-1 (%)':<13} | {'ROUGE-2 (%)':<13} | {'ROUGE-L (%)':<13} | {'Thời gian (s)':<13}")
        print("-" * 80)

        for method in methods:
            metrics = results[group][method]

            if sample_count == 0:
                print(f"{method:<15} | {'N/A':<13} | {'N/A':<13} | {'N/A':<13} | {'N/A':<13}")
                continue

            r1_mean = np.mean(metrics['r1']) * 100
            r2_mean = np.mean(metrics['r2']) * 100
            rl_mean = np.mean(metrics['rl']) * 100
            time_mean = np.mean(metrics['time'])

            # In format dạng bảng
            print(f"{method:<15} | {r1_mean:<13.2f} | {r2_mean:<13.2f} | {rl_mean:<13.2f} | {time_mean:<13.2f}")

        print("-" * 80)

# KÍCH HOẠT (Giả định bạn đã có summarizer và test_data)


In [ ]:
evaluate_summarization_pass1_grouped(summarizer, test_data)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BẮT ĐẦU ĐÁNH GIÁ ROUGE THEO NHÓM ĐỘ DÀI VĂN BẢN...
Đang xử lý Mẫu số 1 (Phân loại: Ngắn)...
Đang xử lý Mẫu số 2 (Phân loại: Ngắn)...
Đang xử lý Mẫu số 3 (Phân loại: Ngắn)...
Đang xử lý Mẫu số 4 (Phân loại: Ngắn)...
Đang xử lý Mẫu số 5 (Phân loại: Ngắn)...
Đang xử lý Mẫu số 6 (Phân loại: Ngắn)...
Đang xử lý Mẫu số 7 (Phân loại: Ngắn)...
Đang xử lý Mẫu số 8 (Phân loại: Ngắn)...
Đang xử lý Mẫu số 9 (Phân loại: Ngắn)...
Đang xử lý Mẫu số 10 (Phân loại: Ngắn)...
Đang xử lý Mẫu số 11 (Phân loại: Trung bình)...
Đang xử lý Mẫu số 12 (Phân loại: Trung bình)...
Đang xử lý Mẫu số 13 (Phân loại: Trung bình)...
Đang xử lý Mẫu số 14 (Phân loại: Trung bình)...
Đang xử lý Mẫu số 15 (Phân loại: Trung bình)...
Đang xử lý Mẫu số 16 (Phân loại: Trung bình)...
Đang xử lý Mẫu số 17 (Phân loại: Trung bình)...
Đang xử lý Mẫu số 18 (Phân loại: Trung bình)...
Đang xử lý Mẫu số 19 (Phân loại: Trung bình)...
Đang xử lý Mẫu số 20 (Phân loại: Trung bình)...
Đang xử lý Mẫu số 21 (Phân loại: Dài)...
Đang xử lý Mẫu số

In [ ]:
# Cell 1: Tạo thư mục và 3 file transcript mẫu (bạn có thể thay nội dung sau)
import os

os.makedirs("/content/tune_summarizer", exist_ok=True)

# Tạo 3 transcript mẫu (bạn có thể thay bằng transcript thật của mình sau)
with open("/content/tune_summarizer/meeting1.txt", "w", encoding="utf-8") as f:
    f.write("""Cuộc họp sáng nay bắt đầu lúc 8h30. Anh A báo cáo doanh số tháng 3 giảm 12% so với tháng trước. Chị B cho rằng cần đẩy mạnh quảng cáo trên TikTok và Facebook. Mọi người thảo luận khoảng 15 phút và thống nhất thử chiến dịch 2 tuần. Anh C phụ trách theo dõi kết quả.""")

with open("/content/tune_summarizer/meeting2.txt", "w", encoding="utf-8") as f:
    f.write("""Chào các bạn. Hôm nay chúng ta bàn về dự án website mới. Anh D nói backend gặp vấn đề về tốc độ. Chị E đề xuất dùng Redis để cache. Sau khi thảo luận, mọi người đồng ý điều chỉnh timeline, đẩy thời gian ra mắt sang tuần thứ 4 tháng sau. Cuối cùng ghi nhận ý kiến của anh F về giao diện.""")

with open("/content/tune_summarizer/meeting3.txt", "w", encoding="utf-8") as f:
    f.write("""Cuộc họp chiều hôm nay khá dài. Đầu tiên là báo cáo tình hình tài chính. Sau đó bàn về tuyển dụng thêm 2 lập trình viên. Anh G lo ngại chi phí cao. Cuối cùng thống nhất phỏng vấn trong tháng này và giao chị H chuẩn bị JD. Cuộc họp kết thúc lúc 17h15.""")

print("Đã tạo 3 file transcript mẫu tại /content/tune_summarizer/")

Đã tạo 3 file transcript mẫu tại /content/tune_summarizer/


In [ ]:
import os

# Tạo thư mục
dir_path = "/content/tune_summarizer/meetings"
os.makedirs(dir_path, exist_ok=True)

print("Đang tạo 15 mẫu transcript dài và đa dạng...\n")

transcripts = [
    # 1. Họp đánh giá doanh số quý
    """Cuộc họp đánh giá kết quả kinh doanh quý 1 năm 2026.\nChủ trì: Anh Hoàng - Giám đốc Kinh doanh. Tham gia: Chị Lan (Marketing), Anh Minh (Sales), Chị Hương (Tài chính), Anh Tuấn (Sản phẩm).\nAnh Hoàng: "Doanh thu quý này chỉ đạt 78% kế hoạch, giảm 22% so với quý trước. Thị trường cạnh tranh rất khốc liệt."\nChị Lan: "Chiến dịch TikTok và Facebook cho chuyển đổi tốt nhưng chi phí quảng cáo tăng 35%. Chúng ta nên tập trung vào video ngắn hơn."\nAnh Minh: "Khách hàng hay so sánh giá với đối thủ. Nhiều người yêu cầu chiết khấu sâu."\nChị Hương lo ngại: "Giảm giá mạnh sẽ ảnh hưởng lợi nhuận. Cần cân nhắc kỹ."\nSau 45 phút thảo luận, thống nhất: Tăng ngân sách marketing online 15%, ra mắt gói khuyến mãi tháng 4, tổ chức training đàm phán cho sales, và họp đánh giá lại sau 3 tuần. Cuộc họp kết thúc lúc 10h50.""",

    # 2. Họp tiến độ dự án website
    """Họp tiến độ dự án website thương mại điện tử.\nAnh Việt (PM) chủ trì. Tham gia: Chị Ngọc (Backend), Anh Khoa (Frontend), Chị Mai (Tester), Anh Long (DevOps).\nAnh Việt: "Dự án chậm 10 ngày. Module thanh toán vẫn chưa xong."\nChị Ngọc: "API VNPay và Momo gặp lỗi, đang debug."\nAnh Khoa: "Frontend đã xong giỏ hàng và checkout."\nChị Mai: "Nên cắt bớt wishlist để kịp deadline."\nQuyết định: Ưu tiên thanh toán trước 25/4, tạm hoãn wishlist, họp daily 15 phút. Cuộc họp kết thúc lúc 11h30.""",

    # 3. Họp tuyển dụng nhân sự
    """Cuộc họp tuyển dụng quý 2.\nChủ trì: Chị Thu - Trưởng phòng HR. Tham gia: Anh Đức (CTO), Chị Vy (Marketing), Anh Bình (Finance).\nChị Thu: "Cần tuyển 3 React Developer và 1 Digital Marketing Specialist."\nAnh Đức: "Yêu cầu React Developer có ít nhất 2 năm kinh nghiệm, biết Redux và Tailwind."\nChị Vy: "Marketing cần người chạy ads Facebook, TikTok và phân tích dữ liệu."\nAnh Bình: "Mức lương đề xuất khá cao, có thể tuyển fresher cho một số vị trí."\nThống nhất: Tuyển 2 Senior React, 1 Mid-level, đăng tin trên TopCV và VietnamWorks, phỏng vấn vòng 1 tuần sau.""",

    # 4. Họp xử lý sự cố server
    """Cuộc họp khẩn về sự cố server down.\nAnh Long chủ trì, toàn đội IT tham gia.\nAnh Long: "Server down gần 2 tiếng, ảnh hưởng hàng nghìn khách."\nAnh Huy: "Do traffic tăng đột biến và database overload."\nChị Linh: "Cần tối ưu query và triển khai Redis caching."\nQuyết định: Nâng cấp cloud, triển khai auto-scaling, backup hàng ngày. Kế hoạch khắc phục trong 1 tuần.""",

    # 5. Họp chiến lược marketing năm mới
    """Họp lập kế hoạch marketing năm 2026.\nChị Lan chủ trì. Tham gia các trưởng nhóm content, ads, SEO.\nChị Lan: "Năm nay mục tiêu tăng traffic 40% và chuyển đổi 25%."\nAnh Kiên (Content): "Nên tập trung vào storytelling và user-generated content."\nChị Phương (Ads): "Tăng tỷ lệ chạy TikTok Shop và livestream."\nThống nhất ngân sách phân bổ: 40% TikTok, 30% Facebook, 20% Google, 10% SEO. Ra mắt campaign lớn vào tháng 6.""",

    # 6. Họp vận hành - tối ưu quy trình làm việc
"""Cuộc họp nội bộ phòng vận hành.\nAnh Hải chủ trì.\nNội dung tập trung vào việc cải thiện quy trình xử lý đơn hàng.\nAnh Hải: "Thời gian xử lý hiện tại còn chậm ở khâu xác nhận."\nChị My: "Nên tự động hóa bước kiểm tra tồn kho."\nAnh Phong: "Cần phối hợp chặt hơn giữa kho và bộ phận giao nhận."\nQuyết định: Áp dụng phần mềm quản lý mới, rút ngắn thời gian xử lý đơn xuống còn 2 giờ.""",


# 7. Họp đào tạo nhân sự mới
"""Cuộc họp xây dựng chương trình onboarding nhân viên.\nChị Vân chủ trì.\nMục tiêu là giúp nhân viên mới hòa nhập nhanh hơn.\nChị Vân: "Nhiều bạn mới còn chưa nắm rõ quy trình công việc."\nAnh Khoa: "Đề xuất bộ tài liệu hướng dẫn chi tiết và video training."\nChị An: "Nên có mentor hỗ trợ trong tháng đầu tiên."\nThống nhất: Thiết kế chương trình đào tạo 2 tuần, phân công mentor cho từng nhân viên mới.""",


# 8. Họp bán hàng - đánh giá doanh số tháng
"""Cuộc họp tổng kết doanh số tháng 4.\nAnh Khánh chủ trì.\nDoanh số tăng 12% so với tháng trước.\nAnh Khánh: "Kết quả tốt nhờ chương trình khuyến mãi cuối tháng."\nChị Hạnh: "Sản phẩm dòng cao cấp bán chậm hơn dự kiến."\nAnh Tài: "Đề xuất ưu đãi thêm cho khách hàng thân thiết."\nQuyết định: Tiếp tục chương trình khuyến mãi, tập trung đẩy mạnh nhóm sản phẩm cao cấp.""",


# 9. Họp dự án ứng dụng di động
"""Cuộc họp cập nhật tiến độ app mobile.\nAnh Nhật chủ trì.\nĐội phát triển báo cáo tiến độ các tính năng chính.\nAnh Nhật: "Tính năng đăng nhập sinh trắc học cần hoàn thiện sớm."\nChị Thảo: "Phần giao diện người dùng đã đạt 90%."\nAnh Duy: "Cần kiểm thử thêm trên hệ điều hành Android."\nThống nhất: Hoàn tất phiên bản beta trong 3 tuần và triển khai test nội bộ.""",

# 10. Họp chiến lược cuối năm
"""Cuộc họp lập kế hoạch chiến lược 6 tháng cuối năm.\nChị Trang chủ trì.\nNội dung xoay quanh mục tiêu tăng trưởng và mở rộng sản phẩm.\nChị Trang: "Cần đặt mục tiêu doanh thu tăng tối thiểu 20%."\nAnh Vũ: "Đề xuất phát triển thêm 2 dòng sản phẩm mới."\nChị Ly: "Ưu tiên ngân sách cho nghiên cứu thị trường."\nQuyết định: Hoàn thiện kế hoạch chi tiết trong 2 tuần, trình ban giám đốc phê duyệt.""",

   # 11. Họp phòng Marketing - chiến dịch mùa hè
"""Cuộc họp triển khai chiến dịch marketing cho mùa hè.\nChị Lan chủ trì.\nMục tiêu chính là tăng nhận diện thương hiệu và doanh số sản phẩm mới.\nChị Lan: "Chiến dịch cần tập trung mạnh vào nền tảng mạng xã hội."\nAnh Phúc: "Đề xuất hợp tác với KOL và chạy quảng cáo video ngắn."\nChị Mai: "Ngân sách cần ưu tiên cho Facebook và TikTok."\nThống nhất: Triển khai chiến dịch trong 6 tuần, ngân sách 300 triệu, đánh giá hiệu quả theo lượt tiếp cận và doanh số hàng tuần.""",


# 12. Họp bộ phận IT - nâng cấp hệ thống
"""Cuộc họp nội bộ phòng công nghệ thông tin.\nAnh Nam chủ trì.\nNội dung chính là nâng cấp hệ thống máy chủ và bảo mật dữ liệu.\nAnh Nam: "Hệ thống hiện tại thường xuyên quá tải vào giờ cao điểm."\nChị Vy: "Cần nâng cấp server và bổ sung tường lửa mới."\nAnh Hùng: "Đề xuất chuyển một phần dữ liệu lên cloud để tối ưu hiệu suất."\nQuyết định: Nâng cấp máy chủ trong tháng này, triển khai sao lưu dữ liệu tự động và rà soát bảo mật định kỳ mỗi tháng.""",


# 13. Họp kinh doanh - kế hoạch mở rộng thị trường
"""Cuộc họp chiến lược kinh doanh quý 3.\nAnh Dũng chủ trì.\nCác thành viên thảo luận về kế hoạch mở rộng thị trường miền Trung.\nAnh Dũng: "Doanh số khu vực này có tiềm năng tăng trưởng cao."\nChị Trâm: "Cần tuyển thêm nhân sự sales tại Đà Nẵng và Huế."\nAnh Long: "Đề xuất chương trình ưu đãi cho đại lý mới."\nThống nhất: Mở thêm 2 văn phòng đại diện, tăng ngân sách bán hàng 15%, bắt đầu triển khai từ đầu tháng 7.""",


# 14. Họp chăm sóc khách hàng
"""Cuộc họp cải thiện chất lượng dịch vụ khách hàng.\nChị Ngọc chủ trì.\nNhiều phản hồi từ khách hàng về thời gian phản hồi chậm.\nChị Ngọc: "Khách hàng đang phải chờ quá lâu để được hỗ trợ."\nAnh Bình: "Nên bổ sung chatbot và tăng ca trực buổi tối."\nChị Hà: "Cần đào tạo lại kỹ năng giao tiếp cho đội ngũ CSKH."\nQuyết định: Triển khai chatbot hỗ trợ 24/7, tuyển thêm 3 nhân sự mới và tổ chức đào tạo nội bộ hàng tháng.""",


# 15. Họp dự án xây dựng website mới
"""Cuộc họp dự án phát triển website công ty.\nAnh Quân chủ trì.\nCác thành viên báo cáo tiến độ thiết kế giao diện và chức năng.\nAnh Quân: "Trang chủ cần tối ưu trải nghiệm người dùng trên điện thoại."\nChị Linh: "Phần thanh toán online đã hoàn thành 80%."\nAnh Sơn: "Cần thêm tính năng theo dõi đơn hàng theo thời gian thực."\nThống nhất: Hoàn thiện giao diện trong 2 tuần, kiểm thử toàn bộ hệ thống trong tuần tiếp theo và dự kiến ra mắt vào đầu tháng sau."""
]

# Tạo file
for i in range(1, 16):
    content = transcripts[i-1]
    full_content = f"""BIÊN BẢN CUỘC HỌP SỐ {i:02d} - {['Doanh số', 'Dự án Website', 'Tuyển dụng', 'Sự cố Server', 'Marketing', 'HR', 'Đối tác', 'Tài chính', 'Sản phẩm mới', 'Đánh giá KPI', 'Khiếu nại KH', 'Kế hoạch năm', 'Cải tiến quy trình', 'Đào tạo', 'Tổng kết năm'][i-1]}\n\n{content}\n\nCuộc họp diễn ra tại văn phòng công ty, bắt đầu lúc 9:00, kết thúc lúc 11:30. Thư ký ghi biên bản: Nguyễn Thị Y."""

    with open(f"{dir_path}/meeting{i:02d}.txt", "w", encoding="utf-8") as f:
        f.write(full_content)

print("✅ Hoàn tất! Đã tạo 15 file transcript tại:", dir_path)
print("Các file có tên: meeting01.txt đến meeting15.txt")
print("Bạn có thể mở thư mục để xem và chỉnh sửa thêm nếu cần.")

Đang tạo 15 mẫu transcript dài và đa dạng...

✅ Hoàn tất! Đã tạo 15 file transcript tại: /content/tune_summarizer/meetings
Các file có tên: meeting01.txt đến meeting15.txt
Bạn có thể mở thư mục để xem và chỉnh sửa thêm nếu cần.


In [ ]:
# Cell 2: Cài đặt thư viện
!pip install -q underthesea transformers torch networkx scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.2 MB/s eta 0:00:00


In [ ]:
# Cell 3: Tuning tự động với 15 mẫu transcript (có LLM Judge + Redundancy Score)

import torch
import re
import os
import numpy as np
from underthesea import sent_tokenize, word_tokenize
from transformers import AutoTokenizer, AutoModelForCausalLM

class MeetingSummarizer:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Device đang sử dụng:", self.device)

        model_path = "/content/drive/MyDrive/Qwen_Model_3B"   # ← thay nếu đường dẫn khác

        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()
        print("Model đã load xong!\n")

    def generate_summary(self, text, max_new_tokens=1000, is_final=True, rep_penalty=1.05):
        if is_final:
            system_prompt = """Bạn là Thư ký cuộc họp. Hãy tóm tắt văn bản thành danh sách gạch đầu dòng phẳng.
<rules>
1. Bắt đầu mỗi ý bằng ký tự "-". KHÔNG dán nhãn chủ đề.
2. KHÔNG ghi các từ mào đầu như: "Vấn đề:", "Ý kiến:", "Kết luận:".
3. TRÌNH TỰ THỜI GIAN: Tóm tắt bao quát toàn bộ sự kiện theo đúng thứ tự.
4. TRÍCH XUẤT TRỰC TIẾP.
</rules>"""
        else:
            system_prompt = """Trích xuất ý chính từ đoạn hội thoại.
<rules>
- Viết dưới dạng gạch đầu dòng (-).
- LỌC RÁC CỰC MẠNH.
- Nếu toàn rác -> PASS
</rules>"""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"<van_ban>\n{text}\n</van_ban>\n\nHãy tóm tắt:"}
        ]

        text_input = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text_input, return_tensors="pt", truncation=True, max_length=2500).to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens if is_final else 200,
                repetition_penalty=rep_penalty,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )

        generated_ids = outputs[0][len(inputs.input_ids[0]):]
        summary = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        summary = summary.replace("<|im_end|>", "").strip()

        if not is_final:
            return summary

        # Post-processing
        cleaned_lines = []
        for line in summary.split('\n'):
            line = line.strip()
            if not line or re.search(r'(chào các bạn|chào mọi người|cảm ơn|hẹn gặp lại)', line, re.IGNORECASE):
                continue
            line = re.sub(r"^(- )?(### Biên bản|Biên bản họp|Tóm tắt|Nội dung chính|Vấn đề).*?:?", "", line, flags=re.IGNORECASE).strip()
            if line and line != "-":
                if not line.startswith('-'):
                    line = '- ' + line
                cleaned_lines.append(line)
        final_summary = '\n'.join(cleaned_lines).strip()
        return final_summary if final_summary else "- Không có nội dung tóm tắt."

    def chunking_method(self, text, chunk_size=10, stride=5, rep_penalty=1.05):
        sentences = sent_tokenize(text)
        if len(sentences) < 6:
            return self.generate_summary(text, is_final=True, rep_penalty=rep_penalty)

        chunks = []
        for i in range(0, len(sentences), stride):
            chunk = " ".join(sentences[i:i + chunk_size])
            chunks.append(chunk)
            if i + chunk_size >= len(sentences):
                break

        partials = []
        for chunk in chunks:
            part = self.generate_summary(chunk, max_new_tokens=200, is_final=False, rep_penalty=rep_penalty)
            if "PASS" not in part and len(part) > 10:
                partials.append(part)

        merged = "\n".join(partials)
        return self.generate_summary(merged, max_new_tokens=1000, is_final=True, rep_penalty=rep_penalty)


# ====================== HÀM ĐÁNH GIÁ TỰ ĐỘNG ======================
def calculate_redundancy_score(summary):
    """Tính điểm chống lặp (cao = tốt)"""
    lines = [line.strip() for line in summary.split('\n') if line.strip().startswith('-')]
    if len(lines) < 2:
        return 1.0

    words_list = [set(word_tokenize(line.lower())) for line in lines]
    duplicate_count = 0
    total_pairs = 0

    for i in range(len(words_list)):
        for j in range(i+1, len(words_list)):
            common = len(words_list[i] & words_list[j])
            if common >= 4:
                duplicate_count += 1
            total_pairs += 1

    redundancy_penalty = duplicate_count / (total_pairs + 1e-8)
    return 1.0 - redundancy_penalty


def llm_judge_score(original_text, summary, summarizer):
    """Dùng model chấm điểm 0-100"""
    prompt = f"""Đánh giá bản tóm tắt cuộc họp sau trên thang điểm 0-100.

Tiêu chí đánh giá:
- Ít lặp từ/câu (redundancy)
- Giữ đúng thứ tự thời gian của sự kiện
- Đầy đủ các ý chính quan trọng
- Câu văn mạch lạc, tự nhiên, chuyên nghiệp

Văn bản gốc:
{original_text[:1800]}

Bản tóm tắt:
{summary}

Chỉ trả về **một số nguyên** từ 0 đến 100, không giải thích thêm."""

    messages = [
        {"role": "system", "content": "Bạn là chuyên gia đánh giá tóm tắt biên bản họp nghiêm khắc và công bằng."},
        {"role": "user", "content": prompt}
    ]

    try:
        text_input = summarizer.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = summarizer.tokenizer(text_input, return_tensors="pt", truncation=True, max_length=2500).to(summarizer.model.device)

        with torch.no_grad():
            outputs = summarizer.model.generate(
                **inputs,
                max_new_tokens=8,
                repetition_penalty=1.15,
                do_sample=False,
                pad_token_id=summarizer.tokenizer.eos_token_id
            )

        result = summarizer.tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
        numbers = re.findall(r'\d+', result)
        score = int(numbers[0]) if numbers else 60
        return min(100, max(30, score))
    except:
        return 50.0


# ====================== CÀI ĐẶT ======================
NUM_SAMPLES = 15                    # Số mẫu đang có
MEETINGS_DIR = "/content/tune_summarizer/meetings"

# ====================== CONFIGS ĐA DẠNG ======================
configs = [
    {"name": "A", "rep_pen": 1.00, "chunk": 8,  "stride": 4, "top_ratio": 0.45, "keep_last": False, "max_final": 1000},
    {"name": "B", "rep_pen": 1.05, "chunk": 8,  "stride": 4, "top_ratio": 0.45, "keep_last": False, "max_final": 1000},
    {"name": "C", "rep_pen": 1.10, "chunk": 8,  "stride": 4, "top_ratio": 0.45, "keep_last": False, "max_final": 1000},
    {"name": "D", "rep_pen": 1.05, "chunk": 6,  "stride": 3, "top_ratio": 0.45, "keep_last": False, "max_final": 1000},
    {"name": "E", "rep_pen": 1.05, "chunk": 9,  "stride": 5, "top_ratio": 0.45, "keep_last": False, "max_final": 1000},
    {"name": "F", "rep_pen": 1.05, "chunk": 10, "stride": 5, "top_ratio": 0.45, "keep_last": False, "max_final": 1000},
    {"name": "G", "rep_pen": 1.05, "chunk": 12, "stride": 6, "top_ratio": 0.45, "keep_last": False, "max_final": 1000},
    {"name": "H", "rep_pen": 1.05, "chunk": 10, "stride": 5, "top_ratio": 0.40, "keep_last": False, "max_final": 1000},
    {"name": "I", "rep_pen": 1.05, "chunk": 10, "stride": 5, "top_ratio": 0.50, "keep_last": False, "max_final": 1000},
    {"name": "J", "rep_pen": 1.05, "chunk": 10, "stride": 5, "top_ratio": 0.45, "keep_last": True,  "max_final": 1000},
    {"name": "K", "rep_pen": 1.05, "chunk": 10, "stride": 5, "top_ratio": 0.45, "keep_last": False, "max_final": 800},
]

# ====================== CHẠY TUNE ======================
summarizer = MeetingSummarizer()
out_dir = "/content/tune_summarizer/results"
os.makedirs(out_dir, exist_ok=True)

print(f"\n=== BẮT ĐẦU TUNE HYPERPARAMETERS TRÊN {NUM_SAMPLES} MẪU ===\n")

config_results = []

for config in configs:
    print(f"Đang chạy Config {config['name']} | rep={config['rep_pen']} | chunk={config['chunk']} | stride={config['stride']}")

    scores = []

    with open(f"{out_dir}/config_{config['name']}.txt", "w", encoding="utf-8") as f:
        f.write(f"CONFIG {config['name']}\n")
        f.write(f"rep_penalty = {config['rep_pen']}\n")
        f.write(f"chunk_size = {config['chunk']}, stride = {config['stride']}\n")
        f.write(f"top_ratio = {config['top_ratio']}, keep_last = {config['keep_last']}\n\n")

        for i in range(1, NUM_SAMPLES + 1):
            txt_file = f"meeting{i:02d}.txt"
            path = f"{MEETINGS_DIR}/{txt_file}"
            with open(path, "r", encoding="utf-8") as tf:
                original_text = tf.read()

            summary = summarizer.chunking_method(
                original_text,
                chunk_size=config["chunk"],
                stride=config["stride"],
                rep_penalty=config["rep_pen"]
            )

            # Tính điểm tự động
            redundancy_score = calculate_redundancy_score(summary)
            judge_score = llm_judge_score(original_text, summary, summarizer)
            final_score = (judge_score * 0.75) + (redundancy_score * 100 * 0.25)

            scores.append(final_score)

            f.write(f"=== {txt_file} ===\n")
            f.write(summary + "\n")
            f.write(f"Judge Score: {judge_score:.1f} | Redundancy: {redundancy_score:.3f} | Final Score: {final_score:.1f}\n\n")

    avg_score = np.mean(scores)
    config_results.append((config['name'], avg_score, config))

    print(f"✓ Config {config['name']} hoàn thành - Average Score: {avg_score:.2f}\n")

# ====================== XẾP HẠNG CONFIG ======================
print("\n" + "="*60)
print("KẾT QUẢ XẾP HẠNG CÁC CONFIG (từ cao đến thấp)")
print("="*60)

config_results.sort(key=lambda x: x[1], reverse=True)

for rank, (name, score, config) in enumerate(config_results, 1):
    print(f"{rank:2d}. Config {name:2s} | Score: {score:.2f} | rep={config['rep_pen']}, chunk={config['chunk']}, stride={config['stride']}")

print("\n" + "="*60)
print(f"✅ Config tốt nhất: {config_results[0][0]} với điểm {config_results[0][1]:.2f}")
print("HOÀN TẤT! Kết quả chi tiết nằm trong thư mục /content/tune_summarizer/results/")

Device đang sử dụng: cuda


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model đã load xong!


=== BẮT ĐẦU TUNE HYPERPARAMETERS TRÊN 15 MẪU ===

Đang chạy Config A | rep=1.0 | chunk=8 | stride=4
✓ Config A hoàn thành - Average Score: 60.55

Đang chạy Config B | rep=1.05 | chunk=8 | stride=4
✓ Config B hoàn thành - Average Score: 60.73

Đang chạy Config C | rep=1.1 | chunk=8 | stride=4
✓ Config C hoàn thành - Average Score: 60.58

Đang chạy Config D | rep=1.05 | chunk=6 | stride=3
✓ Config D hoàn thành - Average Score: 60.03

Đang chạy Config E | rep=1.05 | chunk=9 | stride=5
✓ Config E hoàn thành - Average Score: 59.91

Đang chạy Config F | rep=1.05 | chunk=10 | stride=5
✓ Config F hoàn thành - Average Score: 59.59

Đang chạy Config G | rep=1.05 | chunk=12 | stride=6
✓ Config G hoàn thành - Average Score: 60.27

Đang chạy Config H | rep=1.05 | chunk=10 | stride=5
✓ Config H hoàn thành - Average Score: 59.59

Đang chạy Config I | rep=1.05 | chunk=10 | stride=5
✓ Config I hoàn thành - Average Score: 59.59

Đang chạy Config J | rep=1.05 | chunk=10 | stride=5
✓